## One-Time Preprocessing

In [1]:
import private_info
from pathlib import Path
import pandas as pd
import gzip
import csv

BASE = Path(private_info.path_to_data)
MIMIC = BASE / "mimiciv" / "3.1"
NOTES = BASE / "mimic-iv-note" / "2.2"
ED = BASE / "mimic-iv-ed" / "2.2"

TARGET_DIRS = {
    "hosp": MIMIC / "hosp",
    "icu": MIMIC / "icu",
    "notes": NOTES / "note",
    "ed": ED / "ed"
}

def read_header(csv_path: Path):
    if csv_path.suffixes[-2:] == [".csv", ".gz"]:
        with gzip.open(csv_path, "rt") as f:
            return next(csv.reader(f))
    elif csv_path.suffix == ".csv":
        with open(csv_path, "r") as f:
            return next(csv.reader(f))
    else:
        return None

for domain, folder in TARGET_DIRS.items():
    print("=" * 120)
    print(f"DOMAIN: {domain}")
    print("=" * 120)

    for file in sorted(folder.glob("*.csv*")):
        header = read_header(file)
        if header is None:
            continue

        print(f"\nFILE: {file.name}")
        print(f"N_COLUMNS: {len(header)}")
        print("COLUMNS:")
        for col in header:
            print(f"  - {col}")

DOMAIN: hosp

FILE: admissions.csv.gz
N_COLUMNS: 16
COLUMNS:
  - subject_id
  - hadm_id
  - admittime
  - dischtime
  - deathtime
  - admission_type
  - admit_provider_id
  - admission_location
  - discharge_location
  - insurance
  - language
  - marital_status
  - race
  - edregtime
  - edouttime
  - hospital_expire_flag

FILE: d_hcpcs.csv.gz
N_COLUMNS: 4
COLUMNS:
  - code
  - category
  - long_description
  - short_description

FILE: d_icd_diagnoses.csv.gz
N_COLUMNS: 3
COLUMNS:
  - icd_code
  - icd_version
  - long_title

FILE: d_icd_procedures.csv.gz
N_COLUMNS: 3
COLUMNS:
  - icd_code
  - icd_version
  - long_title

FILE: d_labitems.csv.gz
N_COLUMNS: 4
COLUMNS:
  - itemid
  - label
  - fluid
  - category

FILE: diagnoses_icd.csv.gz
N_COLUMNS: 5
COLUMNS:
  - subject_id
  - hadm_id
  - seq_num
  - icd_code
  - icd_version

FILE: drgcodes.csv.gz
N_COLUMNS: 7
COLUMNS:
  - subject_id
  - hadm_id
  - drg_type
  - drg_code
  - description
  - drg_severity
  - drg_mortality

FILE: emar.csv

In [2]:

import private_info
from pathlib import Path
import pandas as pd

BASE = Path(private_info.path_to_data)
MIMIC = BASE / "mimiciv" / "3.1"
NOTES = BASE / "mimic-iv-note" / "2.2"
ED = BASE / "mimic-iv-ed" / "2.2"

TARGET_DIRS = {
    "hosp": MIMIC / "hosp",
    "icu": MIMIC / "icu",
    "notes": NOTES / "note",
    "ed": ED / "ed"
}

triage_path = TARGET_DIRS["ed"] / "triage.csv.gz"
edstays_path = TARGET_DIRS["ed"] / "edstays.csv.gz"
poe_path = TARGET_DIRS["hosp"] / "poe.csv.gz"

triage_head = pd.read_csv(triage_path, nrows=5)
edstays_head = pd.read_csv(edstays_path, nrows=5)
poe_head = pd.read_csv(poe_path, nrows=5)

print("triage columns:", list(triage_head.columns))
print("edstays columns:", list(edstays_head.columns))
print("poe columns:", list(poe_head.columns))

#  show candidate time columns in POE
time_like = [c for c in poe_head.columns if "time" in c.lower() or "date" in c.lower()]
print("poe time-like columns:", time_like)

#  show a few rows (for understanding key columns)
print("\ntriage head:\n", triage_head)
print("\nedstays head:\n", edstays_head)
print("\npoe head:\n", poe_head)


triage columns: ['subject_id', 'stay_id', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint']
edstays columns: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']
poe columns: ['poe_id', 'poe_seq', 'subject_id', 'hadm_id', 'ordertime', 'order_type', 'order_subtype', 'transaction_type', 'discontinue_of_poe_id', 'discontinued_by_poe_id', 'order_provider_id', 'order_status']
poe time-like columns: ['ordertime']

triage head:
    subject_id   stay_id  temperature  heartrate  resprate  o2sat    sbp   dbp  \
0    10000032  32952584         97.8       87.0      14.0   97.0   71.0  43.0   
1    10000032  33258284         98.4       70.0      16.0   97.0  106.0  63.0   
2    10000032  35968195         99.4      105.0      18.0   96.0  106.0  57.0   
3    10000032  38112554         98.9       88.0      18.0   97.0  116.0  88.0   
4    10000032  39399961         98.7       77.0      16.0   9

In [3]:
import dask.dataframe as dd

ed_out_dir = TARGET_DIRS["ed"] / "parquet"
ed_out_dir.mkdir(parents=True, exist_ok=True)

triage_dd = dd.read_csv(
    triage_path,
    compression="gzip",
    assume_missing=True,
    blocksize="256MB",
    dtype={"pain": "object"},
)

edstays_dd = dd.read_csv(
    edstays_path,
    compression="gzip",
    assume_missing=True,
    blocksize="256MB",
)

triage_dd.to_parquet(ed_out_dir / "triage.parquet", write_index=False)
edstays_dd.to_parquet(ed_out_dir / "edstays.parquet", write_index=False)

print("Saved ED parquet to:", ed_out_dir)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/io/csv.py:508: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/io/csv.py:508: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(


Saved ED parquet to: /Volumes/Extreme SSD/physionet.org/files/mimic-iv-ed/2.2/ed/parquet


In [4]:
import private_info
from pathlib import Path
import subprocess
import hashlib

import duckdb
from tqdm import tqdm

BASE = Path(private_info.path_to_data)

MIMIC_CSV = BASE / "mimiciv" / "3.1"
NOTES_CSV = BASE / "mimic-iv-note" / "2.2"

MIMIC_PARQUET = BASE / "mimiciv_parquet" / "3.1"
NOTES_PARQUET = BASE / "mimic-iv-note_parquet" / "2.2"

TARGETS = [
    (MIMIC_CSV / "hosp", MIMIC_PARQUET / "hosp"),
    (MIMIC_CSV / "icu",  MIMIC_PARQUET / "icu"),
    (NOTES_CSV / "note", NOTES_PARQUET / "note"),
]


def parquet_name_from_csv(csv_path: Path) -> str:
    n = csv_path.name
    if n.endswith(".csv.gz"):
        return n[:-7] + ".parquet"
    if n.endswith(".csv"):
        return n[:-4] + ".parquet"
    raise ValueError(csv_path)


def count_data_rows_fast(path: Path) -> int:
    # Returns number of data rows (excluding header)
    if path.name.endswith(".csv"):
        out = subprocess.check_output(["wc", "-l", str(path)]).decode("utf-8").strip()
        total_lines = int(out.split()[0])
        return max(0, total_lines - 1)

    if path.name.endswith(".csv.gz"):
        # macOS/Linux: gzip -cd file | wc -l
        p1 = subprocess.Popen(["gzip", "-cd", str(path)], stdout=subprocess.PIPE)
        out = subprocess.check_output(["wc", "-l"], stdin=p1.stdout).decode("utf-8").strip()
        p1.stdout.close()
        p1.wait()
        total_lines = int(out.split()[0])
        return max(0, total_lines - 1)

    raise ValueError(path)


def short_id(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]


# Use on-disk DB + spill dir to keep memory stable
spill_dir = BASE / "_duckdb_spill"
spill_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=str(BASE / "_duckdb_csv2parquet.duckdb"))
con.execute("SET threads = 1")
con.execute("SET preserve_insertion_order = false")  # helps memory :contentReference[oaicite:2]{index=2}
con.execute("SET memory_limit = '8GB'")              # tune: 4GB/8GB/16GB :contentReference[oaicite:3]{index=3}
con.execute(f"PRAGMA temp_directory='{spill_dir.as_posix()}'")
con.execute("PRAGMA enable_progress_bar=false")

# Collect files
all_files: list[tuple[Path, Path]] = []
for src_dir, dst_dir in TARGETS:
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in sorted(src_dir.glob("*.csv*")):
        if p.name.endswith(".csv") or p.name.endswith(".csv.gz"):
            all_files.append((p, dst_dir / parquet_name_from_csv(p)))

for src_path, out_path in tqdm(all_files, desc="CSV -> Parquet (low-RAM)", unit="file"):
    if out_path.exists():
        continue

    total_rows = count_data_rows_fast(src_path)

    # Key: no store_rejects (so nothing huge is stored); still permissive parsing
    # all_varchar avoids "___" -> DOUBLE conversion issues
    # Reduce Parquet row group size to cap memory usage while writing :contentReference[oaicite:4]{index=4}
    con.execute(
        f"""
        COPY (
            SELECT *
            FROM read_csv(
                '{src_path.as_posix()}',
                header=true,
                all_varchar=true,
                strict_mode=false,
                ignore_errors=false,
                quote='"',
                escape='"'
            )
        )
        TO '{out_path.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 50000);
        """
    )

    loaded_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{out_path.as_posix()}')"
    ).fetchone()[0]

    skipped = total_rows - loaded_rows
    print(f"{src_path.name}: total={total_rows:,} loaded={loaded_rows:,} skipped={skipped:,}")

print("Done.")
print(f"MIMIC parquet root: {MIMIC_PARQUET}")
print(f"NOTES parquet root: {NOTES_PARQUET}")
print(f"Spill dir: {spill_dir}")


CSV -> Parquet (low-RAM): 100%|██████████| 35/35 [00:00<00:00, 4138.61file/s]

Done.
MIMIC parquet root: /Volumes/Extreme SSD/physionet.org/files/mimiciv_parquet/3.1
NOTES parquet root: /Volumes/Extreme SSD/physionet.org/files/mimic-iv-note_parquet/2.2
Spill dir: /Volumes/Extreme SSD/physionet.org/files/_duckdb_spill


## Load data

In [5]:
import private_info
from pathlib import Path
import dask.dataframe as dd

BASE = Path(private_info.path_to_data)

MIMIC_PQ = BASE / "mimiciv_parquet" / "3.1"
HOSP_PQ = MIMIC_PQ / "hosp"
ICU_PQ = MIMIC_PQ / "icu"

# -------------------------
# Core cohort + demographics
# -------------------------
patients = dd.read_parquet(
    HOSP_PQ / "patients.parquet",
    columns=["subject_id", "gender", "anchor_age"],
)

patients["subject_id"] = dd.to_numeric(patients["subject_id"], errors="coerce")
patients["anchor_age"] = dd.to_numeric(patients["anchor_age"], errors="coerce")

admissions = dd.read_parquet(
    HOSP_PQ / "admissions.parquet",
    columns=[
        "subject_id", "hadm_id",
        "admittime", "dischtime", "deathtime",
        "race", "insurance", "language",
        "hospital_expire_flag",
    ],
)

admissions["subject_id"] = dd.to_numeric(admissions["subject_id"], errors="coerce")
admissions["hadm_id"] = dd.to_numeric(admissions["hadm_id"], errors="coerce")
admissions["hospital_expire_flag"] = dd.to_numeric(admissions["hospital_expire_flag"], errors="coerce")

admissions["admittime"] = dd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = dd.to_datetime(admissions["dischtime"], errors="coerce")
admissions["deathtime"] = dd.to_datetime(admissions["deathtime"], errors="coerce")

icustays = dd.read_parquet(
    ICU_PQ / "icustays.parquet",
    columns=[
        "subject_id", "hadm_id", "stay_id",
        "intime", "outtime", "los",
        "first_careunit", "last_careunit",
    ],
)

icustays["subject_id"] = dd.to_numeric(icustays["subject_id"], errors="coerce")
icustays["hadm_id"] = dd.to_numeric(icustays["hadm_id"], errors="coerce")
icustays["stay_id"] = dd.to_numeric(icustays["stay_id"], errors="coerce")
icustays["los"] = dd.to_numeric(icustays["los"], errors="coerce")

icustays["intime"] = dd.to_datetime(icustays["intime"], errors="coerce")
icustays["outtime"] = dd.to_datetime(icustays["outtime"], errors="coerce")

# -------------------------
# Health state: vitals + labs
# -------------------------
d_items = dd.read_parquet(
    ICU_PQ / "d_items.parquet",
    columns=["itemid", "label", "category", "unitname"],
)
d_items["itemid"] = dd.to_numeric(d_items["itemid"], errors="coerce")

chartevents = dd.read_parquet(
    ICU_PQ / "chartevents.parquet",
    columns=["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "valuenum", "valueuom"],
)

chartevents["subject_id"] = dd.to_numeric(chartevents["subject_id"], errors="coerce")
chartevents["hadm_id"] = dd.to_numeric(chartevents["hadm_id"], errors="coerce")
chartevents["stay_id"] = dd.to_numeric(chartevents["stay_id"], errors="coerce")
chartevents["itemid"] = dd.to_numeric(chartevents["itemid"], errors="coerce")
chartevents["valuenum"] = dd.to_numeric(chartevents["valuenum"], errors="coerce")
chartevents["charttime"] = dd.to_datetime(chartevents["charttime"], errors="coerce")

d_labitems = dd.read_parquet(
    HOSP_PQ / "d_labitems.parquet",
    columns=["itemid", "label", "category"],
)
d_labitems["itemid"] = dd.to_numeric(d_labitems["itemid"], errors="coerce")

labevents = dd.read_parquet(
    HOSP_PQ / "labevents.parquet",
    columns=["subject_id", "hadm_id", "charttime", "itemid", "valuenum", "valueuom", "flag"],
)

labevents["subject_id"] = dd.to_numeric(labevents["subject_id"], errors="coerce")
labevents["hadm_id"] = dd.to_numeric(labevents["hadm_id"], errors="coerce")
labevents["itemid"] = dd.to_numeric(labevents["itemid"], errors="coerce")
labevents["valuenum"] = dd.to_numeric(labevents["valuenum"], errors="coerce")
labevents["charttime"] = dd.to_datetime(labevents["charttime"], errors="coerce")

# -------------------------
# Decisions (choose what you need)
# -------------------------
procedureevents = dd.read_parquet(
    ICU_PQ / "procedureevents.parquet",
    columns=["subject_id", "hadm_id", "stay_id", "starttime", "endtime", "itemid", "value", "valueuom"],
)

procedureevents["subject_id"] = dd.to_numeric(procedureevents["subject_id"], errors="coerce")
procedureevents["hadm_id"] = dd.to_numeric(procedureevents["hadm_id"], errors="coerce")
procedureevents["stay_id"] = dd.to_numeric(procedureevents["stay_id"], errors="coerce")
procedureevents["itemid"] = dd.to_numeric(procedureevents["itemid"], errors="coerce")
procedureevents["value"] = dd.to_numeric(procedureevents["value"], errors="coerce")

procedureevents["starttime"] = dd.to_datetime(procedureevents["starttime"], errors="coerce")
procedureevents["endtime"] = dd.to_datetime(procedureevents["endtime"], errors="coerce")

inputevents = dd.read_parquet(
    ICU_PQ / "inputevents.parquet",
    columns=[
        "subject_id", "hadm_id", "stay_id",
        "starttime", "endtime", "itemid",
        "amount", "amountuom", "rate", "rateuom",
        "statusdescription",
    ],
)

inputevents["subject_id"] = dd.to_numeric(inputevents["subject_id"], errors="coerce")
inputevents["hadm_id"] = dd.to_numeric(inputevents["hadm_id"], errors="coerce")
inputevents["stay_id"] = dd.to_numeric(inputevents["stay_id"], errors="coerce")
inputevents["itemid"] = dd.to_numeric(inputevents["itemid"], errors="coerce")
inputevents["amount"] = dd.to_numeric(inputevents["amount"], errors="coerce")
inputevents["rate"] = dd.to_numeric(inputevents["rate"], errors="coerce")

inputevents["starttime"] = dd.to_datetime(inputevents["starttime"], errors="coerce")
inputevents["endtime"] = dd.to_datetime(inputevents["endtime"], errors="coerce")

ingredientevents = dd.read_parquet(
    ICU_PQ / "ingredientevents.parquet",
    columns=[
        "subject_id", "hadm_id", "stay_id",
        "starttime", "endtime", "itemid",
        "amount", "amountuom", "rate", "rateuom",
        "statusdescription",
    ],
)

ingredientevents["subject_id"] = dd.to_numeric(ingredientevents["subject_id"], errors="coerce")
ingredientevents["hadm_id"] = dd.to_numeric(ingredientevents["hadm_id"], errors="coerce")
ingredientevents["stay_id"] = dd.to_numeric(ingredientevents["stay_id"], errors="coerce")
ingredientevents["itemid"] = dd.to_numeric(ingredientevents["itemid"], errors="coerce")
ingredientevents["amount"] = dd.to_numeric(ingredientevents["amount"], errors="coerce")
ingredientevents["rate"] = dd.to_numeric(ingredientevents["rate"], errors="coerce")

ingredientevents["starttime"] = dd.to_datetime(ingredientevents["starttime"], errors="coerce")
ingredientevents["endtime"] = dd.to_datetime(ingredientevents["endtime"], errors="coerce")

emar = dd.read_parquet(
    HOSP_PQ / "emar.parquet",
    columns=["subject_id", "hadm_id", "emar_id", "emar_seq", "charttime", "medication", "event_txt", "scheduletime"],
)

emar["subject_id"] = dd.to_numeric(emar["subject_id"], errors="coerce")
emar["hadm_id"] = dd.to_numeric(emar["hadm_id"], errors="coerce")
emar["emar_id"] = dd.to_numeric(emar["emar_id"], errors="coerce")
emar["emar_seq"] = dd.to_numeric(emar["emar_seq"], errors="coerce")

emar["charttime"] = dd.to_datetime(emar["charttime"], errors="coerce")
emar["scheduletime"] = dd.to_datetime(emar["scheduletime"], errors="coerce")

pharmacy = dd.read_parquet(
    HOSP_PQ / "pharmacy.parquet",
    columns=["subject_id", "hadm_id", "pharmacy_id", "starttime", "stoptime", "medication", "route", "status", "verifiedtime"],
)

pharmacy["subject_id"] = dd.to_numeric(pharmacy["subject_id"], errors="coerce")
pharmacy["hadm_id"] = dd.to_numeric(pharmacy["hadm_id"], errors="coerce")
pharmacy["pharmacy_id"] = dd.to_numeric(pharmacy["pharmacy_id"], errors="coerce")

pharmacy["starttime"] = dd.to_datetime(pharmacy["starttime"], errors="coerce")
pharmacy["stoptime"] = dd.to_datetime(pharmacy["stoptime"], errors="coerce")
pharmacy["verifiedtime"] = dd.to_datetime(pharmacy["verifiedtime"], errors="coerce")

prescriptions = dd.read_parquet(
    HOSP_PQ / "prescriptions.parquet",
    columns=["subject_id", "hadm_id", "pharmacy_id", "starttime", "stoptime", "drug", "route"],
)

prescriptions["subject_id"] = dd.to_numeric(prescriptions["subject_id"], errors="coerce")
prescriptions["hadm_id"] = dd.to_numeric(prescriptions["hadm_id"], errors="coerce")
prescriptions["pharmacy_id"] = dd.to_numeric(prescriptions["pharmacy_id"], errors="coerce")

prescriptions["starttime"] = dd.to_datetime(prescriptions["starttime"], errors="coerce")
prescriptions["stoptime"] = dd.to_datetime(prescriptions["stoptime"], errors="coerce")


In [6]:
procedureevents.head()

,subject_id,hadm_id,stay_id,starttime,endtime,itemid,value,valueuom
0,10000032,29079034,39553978,2180-07-23 14:43:00,2180-07-23 14:44:00,225966,1,None
1,10000032,29079034,39553978,2180-07-23 14:24:00,2180-07-23 23:50:00,224275,566,min
2,10000032,29079034,39553978,2180-07-23 14:24:00,2180-07-23 23:50:00,224277,566,min
3,10000690,25860671,37081114,2150-11-02 20:00:00,2150-11-03 13:00:00,224275,1020,min
4,10000690,25860671,37081114,2150-11-02 20:38:00,2150-11-03 11:59:00,224277,921,min


In [7]:
inputevents.head()

,subject_id,hadm_id,stay_id,starttime,endtime,itemid,amount,amountuom,rate,rateuom,statusdescription
0,10000032,29079034,39553978,2180-07-23 17:00:00,2180-07-23 17:01:00,226452,200.0,mL,<NA>,<NA>,FinishedRunning
1,10000032,29079034,39553978,2180-07-23 17:00:00,2180-07-23 17:30:00,220862,49.999999,mL,100,mL/hour,FinishedRunning
2,10000032,29079034,39553978,2180-07-23 17:33:00,2180-07-23 18:03:00,220862,49.999999,mL,100,mL/hour,FinishedRunning
3,10000032,29079034,39553978,2180-07-23 18:56:00,2180-07-23 18:57:00,226452,100.0,mL,<NA>,<NA>,FinishedRunning
4,10000032,29079034,39553978,2180-07-23 21:10:00,2180-07-23 21:11:00,226452,100.0,mL,<NA>,<NA>,FinishedRunning


In [8]:
chartevents.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum,valueuom
0,10000032,29079034,39553978,2180-07-23 12:36:00,226512,39.4,kg
1,10000032,29079034,39553978,2180-07-23 12:36:00,226707,60.0,Inch
2,10000032,29079034,39553978,2180-07-23 12:36:00,226730,152.0,cm
3,10000032,29079034,39553978,2180-07-23 14:00:00,220048,<NA>,<NA>
4,10000032,29079034,39553978,2180-07-23 14:00:00,224642,<NA>,<NA>


In [9]:
d_labitems = d_labitems.compute()
d_labitems.shape

(1650, 3)

In [10]:
d_labitems

,itemid,label,category
0,50801,Alveolar-arterial Gradient,Blood Gas
1,50802,Base Excess,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood Gas
3,50804,Calculated Total CO2,Blood Gas
4,50805,Carboxyhemoglobin,Blood Gas
...,...,...,...
1645,53186,MCH,Chemistry
1646,53187,PAN,Chemistry
1647,53188,Lymphocytes,Chemistry
1648,53189,Platelet Count,Chemistry


In [11]:
d_items = d_items.compute()
d_items.shape

(4095, 4)

In [12]:
d_items

,itemid,label,category,unitname
0,220001,Problem List,General,<NA>
1,220003,ICU Admission date,ADT,<NA>
2,220045,Heart Rate,Routine Vital Signs,bpm
3,220046,Heart rate Alarm - High,Alarms,bpm
4,220047,Heart Rate Alarm - Low,Alarms,bpm
...,...,...,...,...
4090,230172,Patient Reversed,3-Significant Events,None
4091,230173,Patient - Fast Track Protocol,3-Significant Events,None
4092,230174,Nerve block in OR,3-Significant Events,None
4093,230176,IUC Stabilization Device,GI/GU,<NA>


## Data Cleaning

In [13]:
# %%
import pandas as pd
import dask.dataframe as dd

# -----------------------
# Trigger itemids (chartevents)
# -----------------------
MAP_ITEMIDS = [220052, 220181]          # Arterial BP mean, NIBP mean
SPO2_ITEMIDS = [220277]                # O2 saturation pulseoxymetry

MAP_THRESHOLD = 65.0
SPO2_THRESHOLD = 90.0

# -----------------------
# Outcome itemids
# -----------------------
# Vasopressors (inputevents)
PRESSOR_ITEMIDS = [
    221906,  # Norepinephrine
    221289,  # Epinephrine
    222315,  # Vasopressin
    221662,  # Dopamine
    221749,  # Phenylephrine
    229617,  # Epinephrine. (alt)
    229630, 229631, 229632,  # Phenylephrine formulations
    229709, 229764,          # Angiotensin II (Giapreza)
]

# Mechanical ventilation (procedureevents)
VENT_INVASIVE_ITEMIDS = [225792]       # Invasive Ventilation
VENT_NIV_ITEMIDS = [225794]            # Non-invasive Ventilation (optional)

# -----------------------
# Baseline vitals (chartevents)
# -----------------------
VITAL_ITEMIDS = {
    "hr":   [220045],                  # Heart Rate
    "rr":   [220210],                  # Respiratory Rate
    "sbp":  [220050, 220179],          # Arterial SBP, NIBP SBP
    "dbp":  [220051, 220180],          # Arterial DBP, NIBP DBP
    "map":  MAP_ITEMIDS,               # Mean BP
    "spo2": SPO2_ITEMIDS,              # SpO2
    "temp_c": [223762],                # Temperature Celsius
    "temp_f": [223761],                # Temperature Fahrenheit
}

VITAL_LOOKBACK_HOURS = 6

# -----------------------
# Baseline labs (labevents)
# -----------------------
LAB_ITEMIDS = {
    "lactate":    [50813, 52442, 53154],     # Lactate (Blood Gas/Chemistry duplicates)
    "creatinine": [50912, 52546],
    "wbc":        [51301, 51755, 51756],     # "White Blood Cells" (Blood)
    "platelets":  [51265, 53189],            # Platelet Count
    "sodium":     [50983, 52623],
    "potassium":  [50971, 52610],
}

LAB_LOOKBACK_HOURS = 24

# Decision windows
PRESSOR_WINDOW_HOURS = 2
VENT_WINDOW_HOURS = 6

# -----------------------
# Sanity check: decode selected itemids (no keyword search)
# -----------------------
all_itemids = (
    MAP_ITEMIDS
    + SPO2_ITEMIDS
    + PRESSOR_ITEMIDS
    + VENT_INVASIVE_ITEMIDS
    + VENT_NIV_ITEMIDS
    + [i for ids in VITAL_ITEMIDS.values() for i in ids]
)

print("d_items decode (sample):")
print(
    d_items[d_items["itemid"].isin(all_itemids)]
    .drop_duplicates()
    .sort_values(["itemid"])
    .head(30)
)

all_lab_itemids = [i for ids in LAB_ITEMIDS.values() for i in ids]
print("\nd_labitems decode (sample):")
print(
    d_labitems[d_labitems["itemid"].isin(all_lab_itemids)]
    .drop_duplicates()
    .sort_values(["itemid"])
)


d_items decode (sample):
      itemid                                  label             category  \
2     220045                             Heart Rate  Routine Vital Signs   
6     220050       Arterial Blood Pressure systolic  Routine Vital Signs   
7     220051      Arterial Blood Pressure diastolic  Routine Vital Signs   
8     220052           Arterial Blood Pressure mean  Routine Vital Signs   
24    220179   Non Invasive Blood Pressure systolic  Routine Vital Signs   
25    220180  Non Invasive Blood Pressure diastolic  Routine Vital Signs   
26    220181       Non Invasive Blood Pressure mean  Routine Vital Signs   
28    220210                       Respiratory Rate          Respiratory   
36    220277            O2 saturation pulseoxymetry          Respiratory   
280   221289                            Epinephrine          Medications   
293   221662                               Dopamine          Medications   
299   221749                          Phenylephrine          Me

In [14]:

AGE_CUTOFF = 65

cohort = (
    icustays[["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los", "first_careunit", "last_careunit"]]
    .merge(patients[["subject_id", "gender", "anchor_age"]], on="subject_id", how="left")
    .merge(admissions[["subject_id", "hadm_id", "race", "insurance", "language", "hospital_expire_flag"]],
           on=["subject_id", "hadm_id"], how="left")
)

cohort = cohort.assign(
    elderly=cohort["anchor_age"] >= AGE_CUTOFF
).persist()

# Sanity checks
print("cohort head:")
print(cohort.head(5))
print("\ncohort dtypes:")
print(cohort.dtypes)
print("\nunique stay_id (approx via nunique compute):", cohort["stay_id"].nunique().compute())


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+------------------------------+------------+-------------+
| Merge columns                | left dtype | right dtype |
+------------------------------+------------+-------------+
| ('subject_id', 'subject_id') | Int64      | float64     |
| ('hadm_id', 'hadm_id')       | Int64      | float64     |
+------------------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(


cohort head:
   subject_id   hadm_id   stay_id              intime             outtime  \
0    10000032  29079034  39553978 2180-07-23 14:00:00 2180-07-23 23:50:47   
1    10000690  25860671  37081114 2150-11-02 19:37:00 2150-11-06 17:03:17   
2    10000980  26913865  39765666 2189-06-27 08:42:00 2189-06-27 20:38:27   
3    10001217  24597018  37067082 2157-11-20 19:18:02 2157-11-21 22:08:00   
4    10001217  27703517  34592300 2157-12-19 15:42:24 2157-12-20 14:27:41   

        los                       first_careunit  \
0  0.410266   Medical Intensive Care Unit (MICU)   
1  3.893252   Medical Intensive Care Unit (MICU)   
2  0.497535   Medical Intensive Care Unit (MICU)   
3  1.118032  Surgical Intensive Care Unit (SICU)   
4  0.948113  Surgical Intensive Care Unit (SICU)   

                         last_careunit gender  anchor_age  \
0   Medical Intensive Care Unit (MICU)      F          52   
1   Medical Intensive Care Unit (MICU)      F          86   
2   Medical Intensive Care U

In [15]:
def build_first_trigger(chartevents_dd, cohort_dd, trigger_itemids, threshold, trigger_name):
    e = chartevents_dd[["stay_id", "charttime", "itemid", "valuenum"]]
    e = e[e["itemid"].isin(trigger_itemids)]
    e = e[e["valuenum"] < threshold]

    # NEW: restrict charttime to ICU window BEFORE taking min
    stay_window = cohort_dd[["stay_id", "intime", "outtime"]]
    e = e.merge(stay_window, on="stay_id", how="inner")
    e = e[(e["charttime"] >= e["intime"]) & (e["charttime"] <= e["outtime"])]

    t0 = (
        e.groupby("stay_id")["charttime"].min()
        .reset_index().rename(columns={"charttime": "trigger_time"})
    )

    tv = (
        t0.merge(e, on="stay_id", how="left")
        .loc[lambda df: df["charttime"] == df["trigger_time"], ["stay_id", "trigger_time", "valuenum"]]
        .groupby(["stay_id", "trigger_time"])["valuenum"].min()
        .reset_index().rename(columns={"valuenum": "trigger_value"})
    )

    out = (
        cohort_dd.merge(tv, on="stay_id", how="inner")
        .assign(trigger_type=trigger_name)
        .assign(time_since_icu_admission_hours=(lambda df: (df["trigger_time"] - df["intime"]).dt.total_seconds() / 3600.0))
    ).persist()
    return out


map_triggers = build_first_trigger(
    chartevents_dd=chartevents,
    cohort_dd=cohort,
    trigger_itemids=MAP_ITEMIDS,
    threshold=MAP_THRESHOLD,
    trigger_name="MAP_lt_65"
)

spo2_triggers = build_first_trigger(
    chartevents_dd=chartevents,
    cohort_dd=cohort,
    trigger_itemids=SPO2_ITEMIDS,
    threshold=SPO2_THRESHOLD,
    trigger_name="SpO2_lt_90"
)

# Sanity checks
print("MAP triggers rows:", map_triggers.shape[0].compute())
print(map_triggers[["stay_id", "trigger_time", "trigger_value", "time_since_icu_admission_hours"]].head(5))

print("\nSpO2 triggers rows:", spo2_triggers.shape[0].compute())
print(spo2_triggers[["stay_id", "trigger_time", "trigger_value", "time_since_icu_admission_hours"]].head(5))


MAP triggers rows: 69223
    stay_id        trigger_time  trigger_value  time_since_icu_admission_hours
0  39553978 2180-07-23 14:11:00           56.0                        0.183333
1  39698942 2134-12-05 22:01:00           61.0                        3.182500
2  32358465 2131-03-09 23:00:00           54.0                        1.450000
3  31831386 2141-04-21 05:00:00           64.0                       15.653889
4  34389119 2125-06-26 20:00:00           64.0                        1.643611

SpO2 triggers rows: 34042
    stay_id        trigger_time  trigger_value  time_since_icu_admission_hours
0  37081114 2150-11-03 06:00:00           83.0                       10.383333
1  31090461 2130-09-25 23:00:00           88.0                       46.166667
2  32610785 2112-12-02 20:00:00           87.0                       44.600000
3  38392119 2129-06-13 00:44:00           89.0                        0.014444
4  38383343 2137-08-18 09:11:00           87.0                       15.573056


In [16]:
def label_treatment_within_window(triggers_dd, events_dd, event_time_col, event_itemid_col, itemids, window_hours, out_prefix):
    t = triggers_dd[["stay_id", "trigger_time", "outtime"]]

    e = events_dd[["stay_id", event_time_col, event_itemid_col]]
    e = e[e[event_itemid_col].isin(itemids)]

    joined = t.merge(e, on="stay_id", how="left")
    joined = joined.assign(window_end=joined["trigger_time"] + pd.to_timedelta(window_hours, unit="h"))

    in_win = joined[
        (joined[event_time_col] > joined["trigger_time"]) &
        (joined[event_time_col] <= joined["window_end"]) &
        (joined[event_time_col] <= joined["outtime"])   # NEW
    ]

    first_start = (
        in_win.groupby("stay_id")[event_time_col].min()
        .reset_index().rename(columns={event_time_col: f"{out_prefix}_starttime"})
    )

    out = triggers_dd.merge(first_start, on="stay_id", how="left")
    out = out.assign(
        **{f"{out_prefix}_started_within_{window_hours}h": out[f"{out_prefix}_starttime"].notnull()},
        **{f"time_to_{out_prefix}_hours": (out[f"{out_prefix}_starttime"] - out["trigger_time"]).dt.total_seconds() / 3600.0},
    ).persist()
    return out


map_labeled = label_treatment_within_window(
    triggers_dd=map_triggers,
    events_dd=inputevents,
    event_time_col="starttime",
    event_itemid_col="itemid",
    itemids=PRESSOR_ITEMIDS,
    window_hours=PRESSOR_WINDOW_HOURS,
    out_prefix="pressor"
)

spo2_labeled = label_treatment_within_window(
    triggers_dd=spo2_triggers,
    events_dd=procedureevents,
    event_time_col="starttime",
    event_itemid_col="itemid",
    itemids=VENT_INVASIVE_ITEMIDS,
    window_hours=VENT_WINDOW_HOURS,
    out_prefix="vent"
)

# Sanity checks
print("pressor labeled rows:", map_labeled.shape[0].compute())
print(map_labeled[["stay_id", "trigger_time", "pressor_started_within_2h", "time_to_pressor_hours"]].head(5))
print("pressor started rate:", map_labeled["pressor_started_within_2h"].mean().compute())

print("\nvent labeled rows:", spo2_labeled.shape[0].compute())
print(spo2_labeled[["stay_id", "trigger_time", "vent_started_within_6h", "time_to_vent_hours"]].head(5))
print("vent started rate:", spo2_labeled["vent_started_within_6h"].mean().compute())


pressor labeled rows: 69223
    stay_id        trigger_time  pressor_started_within_2h  \
0  39553978 2180-07-23 14:11:00                      False   
1  39698942 2134-12-05 22:01:00                      False   
2  32358465 2131-03-09 23:00:00                      False   
3  31831386 2141-04-21 05:00:00                      False   
4  34389119 2125-06-26 20:00:00                      False   

   time_to_pressor_hours  
0                    NaN  
1                    NaN  
2                    NaN  
3                    NaN  
4                    NaN  
pressor started rate: 0.22018693208904555

vent labeled rows: 34042
    stay_id        trigger_time  vent_started_within_6h  time_to_vent_hours
0  37081114 2150-11-03 06:00:00                   False                 NaN
1  31090461 2130-09-25 23:00:00                   False                 NaN
2  32610785 2112-12-02 20:00:00                   False                 NaN
3  38392119 2129-06-13 00:44:00                   False          

In [17]:
# %%
vital_map_pd = []
for vital, ids in VITAL_ITEMIDS.items():
    for itemid in ids:
        vital_map_pd.append({"itemid": itemid, "vital": vital})
vital_map_pd = pd.DataFrame(vital_map_pd)

vital_map = dd.from_pandas(vital_map_pd, npartitions=1)
ALL_VITAL_ITEMIDS = vital_map_pd["itemid"].tolist()

def build_vitals_wide(triggers_dd, lookback_hours):
    t = triggers_dd[["stay_id", "trigger_time"]].persist()

    v = chartevents[["stay_id", "charttime", "itemid", "valuenum"]]
    v = v[v["itemid"].isin(ALL_VITAL_ITEMIDS)].merge(vital_map, on="itemid", how="inner")

    # Standardize temperature into a single numeric stream "temp"
    v = v.assign(
        vital_std=v["vital"].map_partitions(lambda s: s.replace({"temp_c": "temp", "temp_f": "temp"}), meta=("vital_std", "object"))
    )
    v = v.assign(
        val_std=v["valuenum"]
    )
    # Fahrenheit -> Celsius only for itemid 223761
    v = v.assign(
        val_std=v["val_std"].where(v["itemid"] != 223761, (v["val_std"] - 32.0) * (5.0 / 9.0))
    )

    joined = t.merge(v, on="stay_id", how="left")
    joined = joined.assign(window_start=joined["trigger_time"] - pd.to_timedelta(lookback_hours, unit="h"))
    joined = joined[(joined["charttime"] >= joined["window_start"]) & (joined["charttime"] < joined["trigger_time"])]

    agg = (
        joined.groupby(["stay_id", "vital_std"])["val_std"]
        .agg(["mean", "min", "max"])
        .reset_index()
    )

    last_t = (
        joined.groupby(["stay_id", "vital_std"])["charttime"]
        .max()
        .reset_index()
        .rename(columns={"charttime": "last_charttime"})
    )

    last_v = (
        joined.merge(last_t, on=["stay_id", "vital_std"], how="inner")
        .loc[lambda df: df["charttime"] == df["last_charttime"], ["stay_id", "vital_std", "val_std"]]
        .groupby(["stay_id", "vital_std"])["val_std"]
        .max()
        .reset_index()
        .rename(columns={"val_std": "last"})
    )

    stats = agg.merge(last_v, on=["stay_id", "vital_std"], how="left")

    # Make wide by merging per vital (no pivot/search)
    out = None
    vitals = ["hr", "rr", "sbp", "dbp", "map", "spo2", "temp"]
    for vi in vitals:
        part = stats[stats["vital_std"] == vi][["stay_id", "last", "mean", "min", "max"]]
        part = part.rename(columns={
            "last": f"{vi}_last",
            "mean": f"{vi}_mean",
            "min": f"{vi}_min",
            "max": f"{vi}_max",
        })
        out = part if out is None else out.merge(part, on="stay_id", how="outer")

    return out

map_vitals = build_vitals_wide(map_labeled, VITAL_LOOKBACK_HOURS).persist()
spo2_vitals = build_vitals_wide(spo2_labeled, VITAL_LOOKBACK_HOURS).persist()

# Sanity checks
print("map_vitals head:")
print(map_vitals.head(5))
print("non-null hr_last rate (MAP triggers):", map_vitals["hr_last"].notnull().mean().compute())

print("\nspo2_vitals head:")
print(spo2_vitals.head(5))
print("non-null spo2_last rate (SpO2 triggers):", spo2_vitals["spo2_last"].notnull().mean().compute())


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------------+------------+-------------+
| Merge columns        | left dtype | right dtype |
+----------------------+------------+-------------+
| ('itemid', 'itemid') | Int64      | int64       |
+----------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------------+------------+-------------+
| Merge columns        | left dtype | right dtype |
+----------------------+------------+-------------+
| ('itemid', 'itemid') | Int64      | int64       |
+----------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected resu

map_vitals head:
    stay_id  hr_last    hr_mean  hr_min  hr_max  rr_last    rr_mean  rr_min  \
0  30000646     89.0  95.428571    87.0   102.0     32.0       26.0    18.0   
1  30002521     84.0  93.833333    84.0   100.0     14.0  18.666667    11.0   
2  30003125     59.0  69.142857    59.0    79.0     18.0  20.285714    18.0   
3  30004320     74.0       80.0    74.0    85.0     11.0     15.375     9.0   
4  30005160     68.0  74.428571    68.0    78.0     16.0  16.142857    15.0   

   rr_max  sbp_last  ...  map_min  map_max  spo2_last  spo2_mean  spo2_min  \
0    33.0      98.0  ...     67.0     80.0       91.0  96.285714      91.0   
1    27.0     140.0  ...     73.0    118.0       99.0  98.833333      97.0   
2    24.0     138.0  ...     74.0     90.0       96.0  95.285714      94.0   
3    22.0      90.0  ...     69.0     98.0       98.0     96.375      95.0   
4    17.0     113.0  ...     66.0     78.0       98.0       94.0      88.0   

   spo2_max  temp_last  temp_mean   tem

In [18]:
# %%
lab_map_pd = []
for lab, ids in LAB_ITEMIDS.items():
    for itemid in ids:
        lab_map_pd.append({"itemid": itemid, "lab": lab})
lab_map_pd = pd.DataFrame(lab_map_pd)

lab_map = dd.from_pandas(lab_map_pd, npartitions=1)
ALL_LAB_ITEMIDS = lab_map_pd["itemid"].tolist()

import pandas as pd

def build_labs_wide(triggers_dd, lookback_hours):
    #  base trigger table
    t = triggers_dd[["stay_id", "hadm_id", "trigger_time"]].persist()

    #  filter relevant lab itemids and map itemid -> lab
    labs = labevents[["hadm_id", "charttime", "itemid", "valuenum", "valueuom"]]
    labs = labs[labs["itemid"].isin(ALL_LAB_ITEMIDS)].merge(lab_map, on="itemid", how="inner")

    #  join and apply lookback window
    joined = t.merge(labs, on="hadm_id", how="left")
    joined = joined.assign(window_start=joined["trigger_time"] - pd.to_timedelta(lookback_hours, unit="h"))
    joined = joined[(joined["charttime"] >= joined["window_start"]) & (joined["charttime"] < joined["trigger_time"])]

    #  last lab time must be max(charttime), NOT mean(charttime)
    last_t = (
        joined.groupby(["stay_id", "lab"])
        .agg({"charttime": "max"})
        .reset_index()
        .rename(columns={"charttime": "last_charttime"})
    )

    #  keep only rows at the last timestamp
    last_rows = (
        joined.merge(last_t, on=["stay_id", "lab"], how="inner")
        .loc[lambda df: df["charttime"] == df["last_charttime"], ["stay_id", "lab", "valuenum", "valueuom"]]
    )

    #  avoid mixing units at the last timestamp
    # Pick the most frequent valueuom within (stay_id, lab, last_time).
    unit_counts = (
        last_rows.groupby(["stay_id", "lab", "valueuom"])
        .size()
        .reset_index()
        .rename(columns={0: "n"})
    )

    unit_max = (
        unit_counts.groupby(["stay_id", "lab"])
        .agg({"n": "max"})
        .reset_index()
        .rename(columns={"n": "n_max"})
    )

    keep_units = (
        unit_counts.merge(unit_max, on=["stay_id", "lab"], how="inner")
        .loc[lambda df: df["n"] == df["n_max"], ["stay_id", "lab", "valueuom"]]
        .drop_duplicates()
    )

    last_rows_u = last_rows.merge(keep_units, on=["stay_id", "lab", "valueuom"], how="inner")

    #  if still multiple rows (same lab/unit/time), average them
    last_v = (
        last_rows_u.groupby(["stay_id", "lab"])
        .agg({"valuenum": "mean"})
        .reset_index()
    )

    out = None
    for lab in LAB_ITEMIDS.keys():
        part = last_v[last_v["lab"] == lab][["stay_id", "valuenum"]].rename(columns={"valuenum": f"{lab}_last"})
        out = part if out is None else out.merge(part, on="stay_id", how="outer")

    return out

map_labs = build_labs_wide(map_labeled, LAB_LOOKBACK_HOURS).persist()
spo2_labs = build_labs_wide(spo2_labeled, LAB_LOOKBACK_HOURS).persist()

# Sanity checks
print("map_labs head:")
print(map_labs.head(5))
print("non-null lactate_last rate (MAP triggers):", map_labs["lactate_last"].notnull().mean().compute())

print("\nspo2_labs head:")
print(spo2_labs.head(5))
print("non-null creatinine_last rate (SpO2 triggers):", spo2_labs["creatinine_last"].notnull().mean().compute())


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------------+------------+-------------+
| Merge columns        | left dtype | right dtype |
+----------------------+------------+-------------+
| ('itemid', 'itemid') | Int64      | int64       |
+----------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------------+------------+-------------+
| Merge columns        | left dtype | right dtype |
+----------------------+------------+-------------+
| ('itemid', 'itemid') | Int64      | int64       |
+----------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected resu

map_labs head:
    stay_id  lactate_last  creatinine_last  wbc_last  platelets_last  \
0  30000646          <NA>              0.9       8.5           268.0   
1  30001148           1.5              0.6      10.9           177.0   
2  30001947           0.7              0.8       5.0           161.0   
3  30002521          <NA>              0.6      <NA>            <NA>   
4  30003125          <NA>              0.7       9.9           150.0   

   sodium_last  potassium_last  
0        138.0             3.5  
1         <NA>            <NA>  
2        140.0             3.9  
3        133.0             7.6  
4        139.0             3.6  
non-null lactate_last rate (MAP triggers): 0.6079059258863924

spo2_labs head:
    stay_id  lactate_last  creatinine_last  wbc_last  platelets_last  \
0  30000646          <NA>              0.9       8.5           268.0   
1  30000831           1.4              2.2      14.2           285.0   
2  30001471          <NA>              0.9       7.2       

In [19]:
# %%
analytic_pressor = (
    map_labeled[
        [
            "subject_id", "hadm_id", "stay_id",
            "intime", "outtime",
            "trigger_type", "trigger_time", "trigger_value", "time_since_icu_admission_hours",
            "anchor_age", "elderly", "gender", "race", "insurance", "language",
            "first_careunit", "last_careunit",
            "hospital_expire_flag",
            "pressor_started_within_2h", "time_to_pressor_hours",
        ]
    ]
    .merge(map_vitals, on="stay_id", how="left")
    .merge(map_labs, on="stay_id", how="left")
).persist()

analytic_vent = (
    spo2_labeled[
        [
            "subject_id", "hadm_id", "stay_id",
            "intime", "outtime",
            "trigger_type", "trigger_time", "trigger_value", "time_since_icu_admission_hours",
            "anchor_age", "elderly", "gender", "race", "insurance", "language",
            "first_careunit", "last_careunit",
            "hospital_expire_flag",
            "vent_started_within_6h", "time_to_vent_hours",
        ]
    ]
    .merge(spo2_vitals, on="stay_id", how="left")
    .merge(spo2_labs, on="stay_id", how="left")
).persist()

# Sanity checks
print("analytic_pressor head:")
print(analytic_pressor.head(3))
print("analytic_pressor rows:", analytic_pressor.shape[0].compute())
print("elderly rate (pressor dataset):", analytic_pressor["elderly"].mean().compute())

print("\nanalytic_vent head:")
print(analytic_vent.head(3))
print("analytic_vent rows:", analytic_vent.shape[0].compute())
print("elderly rate (vent dataset):", analytic_vent["elderly"].mean().compute())


analytic_pressor head:
   subject_id   hadm_id   stay_id              intime             outtime  \
0    10005593  26835370  34389119 2125-06-26 18:21:23 2125-06-29 20:51:35   
1    10018081  28861356  38333427 2134-08-05 14:53:33 2134-08-07 17:32:43   
2    10020740  23831430  35026312 2150-03-11 15:34:56 2150-03-19 02:17:47   

  trigger_type        trigger_time  trigger_value  \
0    MAP_lt_65 2125-06-26 20:00:00           64.0   
1    MAP_lt_65 2134-08-05 16:11:00           -7.0   
2    MAP_lt_65 2150-03-12 01:00:00           60.0   

   time_since_icu_admission_hours  anchor_age  ...  temp_last  temp_mean  \
0                        1.643611          61  ...       36.1       36.1   
1                        1.290833          79  ...  37.055556  37.055556   
2                        9.417778          56  ...  37.611111  38.092593   

    temp_min   temp_max lactate_last creatinine_last wbc_last  platelets_last  \
0       36.1       36.1          2.1             2.9     16.1        

In [20]:
# %%
print("pressor: rows", analytic_pressor.shape[0].compute(),
      "unique stay_id", analytic_pressor["stay_id"].nunique().compute())

print("vent: rows", analytic_vent.shape[0].compute(),
      "unique stay_id", analytic_vent["stay_id"].nunique().compute())


pressor: rows 69223 unique stay_id 69223
vent: rows 34042 unique stay_id 34042


In [21]:
# %%
bad_pressor = ((analytic_pressor["trigger_time"] < analytic_pressor["intime"]) |
               (analytic_pressor["trigger_time"] > analytic_pressor["outtime"])).mean().compute()

bad_vent = ((analytic_vent["trigger_time"] < analytic_vent["intime"]) |
            (analytic_vent["trigger_time"] > analytic_vent["outtime"])).mean().compute()

print("pressor: trigger outside ICU rate:", bad_pressor)
print("vent: trigger outside ICU rate:", bad_vent)


pressor: trigger outside ICU rate: 0.0
vent: trigger outside ICU rate: 0.0


In [22]:
# %%
vital_cols = ["hr_last","map_last","spo2_last","rr_last","temp_last"]
lab_cols = ["lactate_last","creatinine_last","wbc_last","platelets_last","sodium_last","potassium_last"]

print("pressor vitals missingness:")
print(analytic_pressor[vital_cols].isnull().mean().compute().sort_values())

print("\npressor labs missingness:")
print(analytic_pressor[lab_cols].isnull().mean().compute().sort_values())

print("\nvent vitals missingness:")
print(analytic_vent[vital_cols].isnull().mean().compute().sort_values())

print("\nvent labs missingness:")
print(analytic_vent[lab_cols].isnull().mean().compute().sort_values())


pressor vitals missingness:
hr_last      0.067348
rr_last      0.074383
spo2_last    0.090071
map_last     0.151106
temp_last    0.195788
dtype: float64

pressor labs missingness:
platelets_last     0.240354
wbc_last           0.242622
creatinine_last    0.249411
potassium_last     0.270185
sodium_last        0.275429
lactate_last       0.506363
dtype: float64

vent vitals missingness:
hr_last      0.050467
rr_last      0.055931
map_last     0.090065
spo2_last    0.094472
temp_last    0.152194
dtype: float64

vent labs missingness:
creatinine_last    0.147494
potassium_last     0.150461
sodium_last        0.151636
platelets_last     0.156953
wbc_last           0.157423
lactate_last       0.560513
dtype: float64


In [23]:
# %%
import pandas as pd

pe = inputevents[["stay_id", "starttime", "endtime", "itemid"]]
pe = pe[pe["itemid"].isin(PRESSOR_ITEMIDS)]

pj = analytic_pressor[["stay_id", "trigger_time"]].merge(pe, on="stay_id", how="left")

pressor_active_row = (
    (pj["starttime"] <= pj["trigger_time"]) &
    (pj["endtime"].isnull() | (pj["endtime"] > pj["trigger_time"]))
)

pressor_active_by_stay = (
    pj.assign(active_int=pressor_active_row.astype("int8"))
      .groupby("stay_id")["active_int"]
      .max()
      .reset_index()
      .rename(columns={"active_int": "pressor_active_at_t0"})
)

analytic_pressor = analytic_pressor.merge(pressor_active_by_stay, on="stay_id", how="left")
analytic_pressor = analytic_pressor.assign(
    pressor_active_at_t0=analytic_pressor["pressor_active_at_t0"].fillna(0).astype("int8")
).persist()

# Sanity checks
print("pressor_active_at_t0 rate:", analytic_pressor["pressor_active_at_t0"].mean().compute())
print("pressor_active_at_t0 value counts:")
print(analytic_pressor["pressor_active_at_t0"].value_counts().compute())

print("\npreview:")
print(analytic_pressor[["stay_id", "trigger_time", "pressor_started_within_2h", "pressor_active_at_t0"]].head(5))


pressor_active_at_t0 rate: 0.15947011831327737
pressor_active_at_t0 value counts:
pressor_active_at_t0
1    11039
0    58184
Name: count, dtype: int64

preview:
    stay_id        trigger_time  pressor_started_within_2h  \
0  34389119 2125-06-26 20:00:00                      False   
1  38333427 2134-08-05 16:11:00                      False   
2  35026312 2150-03-12 01:00:00                      False   
3  33683112 2189-06-09 19:00:00                      False   
4  30757476 2131-02-28 01:00:00                      False   

   pressor_active_at_t0  
0                     0  
1                     0  
2                     0  
3                     0  
4                     1  


In [24]:
# %%
ve = procedureevents[["stay_id", "starttime", "endtime", "itemid"]]
ve = ve[ve["itemid"].isin(VENT_INVASIVE_ITEMIDS)]

vj = analytic_vent[["stay_id", "trigger_time"]].merge(ve, on="stay_id", how="left")

vent_active_row = (
    (vj["starttime"] <= vj["trigger_time"]) &
    (vj["endtime"].isnull() | (vj["endtime"] > vj["trigger_time"]))
)

vent_active_by_stay = (
    vj.assign(active_int=vent_active_row.astype("int8"))
      .groupby("stay_id")["active_int"]
      .max()
      .reset_index()
      .rename(columns={"active_int": "vent_active_at_t0"})
)

analytic_vent = analytic_vent.merge(vent_active_by_stay, on="stay_id", how="left")
analytic_vent = analytic_vent.assign(
    vent_active_at_t0=analytic_vent["vent_active_at_t0"].fillna(0).astype("int8")
).persist()

# Sanity checks
print("vent_active_at_t0 rate:", analytic_vent["vent_active_at_t0"].mean().compute())
print("vent_active_at_t0 value counts:")
print(analytic_vent["vent_active_at_t0"].value_counts().compute())

print("\npreview:")
print(analytic_vent[["stay_id", "trigger_time", "vent_started_within_6h", "vent_active_at_t0"]].head(5))


vent_active_at_t0 rate: 0.16218201045766995
vent_active_at_t0 value counts:
vent_active_at_t0
1     5521
0    28521
Name: count, dtype: int64

preview:
    stay_id        trigger_time  vent_started_within_6h  vent_active_at_t0
0  31959184 2110-12-01 05:00:00                   False                  0
1  33683112 2189-06-09 12:55:00                   False                  0
2  38197705 2116-12-07 11:25:00                   False                  1
3  38554095 2175-03-27 04:00:00                   False                  0
4  37036476 2187-02-27 22:28:00                   False                  0


In [25]:
# %%
analytic_pressor = analytic_pressor.assign(
    pressor_initiated_within_2h=(analytic_pressor["pressor_started_within_2h"] & (analytic_pressor["pressor_active_at_t0"] == 0))
).persist()

analytic_vent = analytic_vent.assign(
    vent_initiated_within_6h=(analytic_vent["vent_started_within_6h"] & (analytic_vent["vent_active_at_t0"] == 0))
).persist()

# Sanity checks
print("pressor initiated within 2h rate:", analytic_pressor["pressor_initiated_within_2h"].mean().compute())
print("vent initiated within 6h rate:", analytic_vent["vent_initiated_within_6h"].mean().compute())

print("\npressor cross-tab (started vs active):")
print(
    analytic_pressor.groupby(["pressor_started_within_2h", "pressor_active_at_t0"])["stay_id"]
    .count()
    .compute()
)

print("\nvent cross-tab (started vs active):")
print(
    analytic_vent.groupby(["vent_started_within_6h", "vent_active_at_t0"])["stay_id"]
    .count()
    .compute()
)


pressor initiated within 2h rate: 0.0957196307585629
vent initiated within 6h rate: 0.04838141119793197

pressor cross-tab (started vs active):
pressor_started_within_2h  pressor_active_at_t0
False                      0                       51558
                           1                        2423
True                       0                        6626
                           1                        8616
Name: stay_id, dtype: Int64

vent cross-tab (started vs active):
vent_started_within_6h  vent_active_at_t0
False                   0                    26874
                        1                     5501
True                    0                     1647
                        1                       20
Name: stay_id, dtype: Int64


In [26]:
# %%
import numpy as np

def add_fixed_horizon_survival_columns(
    df,
    *,
    time_col,
    active_col,
    event_col,
    tau_hours,
):
    in_risk = (df[active_col] == 0).astype("int8")
    event = ((df[event_col] == 1) & (in_risk == 1)).astype("int8")

    icu_follow_up_hours = ((df["outtime"] - df["trigger_time"]).dt.total_seconds() / 3600.0).astype("float64")
    icu_follow_up_hours = icu_follow_up_hours.where(icu_follow_up_hours >= 0, 0.0)

    follow_up_hours = icu_follow_up_hours.where(icu_follow_up_hours <= float(tau_hours), float(tau_hours))
    follow_up_hours = follow_up_hours.where(in_risk == 1)

    event_time_hours = df[time_col].where(event == 1).astype("float64")
    duration_hours = follow_up_hours.where(event == 0, event_time_hours).astype("float64")

    censored = ((in_risk == 1) & (event == 0)).astype("int8")
    censor_admin_tau = ((in_risk == 1) & (event == 0) & (follow_up_hours == float(tau_hours))).astype("int8")
    censor_icu_discharge = ((in_risk == 1) & (event == 0) & (follow_up_hours < float(tau_hours))).astype("int8")

    return df.assign(
        surv_tau_hours=np.float64(tau_hours),
        surv_in_risk=in_risk,
        surv_event=event,
        surv_censored=censored,
        surv_event_time_hours=event_time_hours,
        surv_icu_follow_up_hours=icu_follow_up_hours.astype("float64"),
        surv_follow_up_hours=follow_up_hours.astype("float64"),
        surv_duration_hours=duration_hours,
        surv_censor_admin_tau=censor_admin_tau,
        surv_censor_icu_discharge=censor_icu_discharge,
    ).persist()


analytic_pressor = add_fixed_horizon_survival_columns(
    analytic_pressor,
    time_col="time_to_pressor_hours",
    active_col="pressor_active_at_t0",
    event_col="pressor_initiated_within_2h",
    tau_hours=2.0,
)

analytic_vent = add_fixed_horizon_survival_columns(
    analytic_vent,
    time_col="time_to_vent_hours",
    active_col="vent_active_at_t0",
    event_col="vent_initiated_within_6h",
    tau_hours=6.0,
)

# Sanity checks
print("pressor surv_in_risk rate:", analytic_pressor["surv_in_risk"].mean().compute())
print("pressor surv_event rate:", analytic_pressor["surv_event"].mean().compute())
print("pressor surv_censored rate:", analytic_pressor["surv_censored"].mean().compute())
print(analytic_pressor[[
    "stay_id", "pressor_active_at_t0", "pressor_initiated_within_2h",
    "surv_in_risk", "surv_event", "surv_follow_up_hours", "surv_duration_hours"
]].head(5))

print("\nvent surv_in_risk rate:", analytic_vent["surv_in_risk"].mean().compute())
print("vent surv_event rate:", analytic_vent["surv_event"].mean().compute())
print("vent surv_censored rate:", analytic_vent["surv_censored"].mean().compute())
print(analytic_vent[[
    "stay_id", "vent_active_at_t0", "vent_initiated_within_6h",
    "surv_in_risk", "surv_event", "surv_follow_up_hours", "surv_duration_hours"
]].head(5))


pressor surv_in_risk rate: 0.8405298816867226
pressor surv_event rate: 0.0957196307585629
pressor surv_censored rate: 0.7448102509281597
    stay_id  pressor_active_at_t0  pressor_initiated_within_2h  surv_in_risk  \
0  34389119                     0                        False             1   
1  38333427                     0                        False             1   
2  35026312                     0                        False             1   
3  33683112                     0                        False             1   
4  30757476                     1                        False             0   

   surv_event  surv_follow_up_hours  surv_duration_hours  
0           0                   2.0                  2.0  
1           0                   2.0                  2.0  
2           0                   2.0                  2.0  
3           0                   2.0                  2.0  
4           0                   NaN                  NaN  

vent surv_in_risk rate: 0.8

In [27]:
# %%
import pandas as pd

adm_death = admissions[["subject_id", "hadm_id", "dischtime", "deathtime", "hospital_expire_flag"]]

def add_inhospital_death_features(df):
    # suffixes=('', '_adm') keeps existing columns intact and puts admissions columns under *_adm if collisions happen
    df = df.merge(adm_death, on=["subject_id", "hadm_id"], how="left", suffixes=("", "_adm"))

    # Normalize dischtime / deathtime (prefer existing if present, else use *_adm)
    for col in ["dischtime", "deathtime", "hospital_expire_flag"]:
        alt = f"{col}_adm"
        if (col not in df.columns) and (alt in df.columns):
            df = df.rename(columns={alt: col})
        elif (col in df.columns) and (alt in df.columns):
            df = df.assign(**{
                col: df[col].where(df[col].notnull(), df[alt])
            }).drop(columns=[alt])

    death_event_in_hosp = (
        df["deathtime"].notnull() &
        (df["deathtime"] > df["trigger_time"]) &
        (df["deathtime"] <= df["dischtime"])
    )

    event_or_censor_time = df["deathtime"].where(death_event_in_hosp, df["dischtime"])
    time_to_death_hours = (event_or_censor_time - df["trigger_time"]).dt.total_seconds() / 3600.0

    df = df.assign(
        death_event_in_hosp=death_event_in_hosp.astype("int8"),
        time_to_death_hours=time_to_death_hours.astype("float64"),
        death_within_7d_of_trigger=(death_event_in_hosp & (time_to_death_hours <= 7 * 24)).astype("int8"),
        death_within_30d_of_trigger=(death_event_in_hosp & (time_to_death_hours <= 30 * 24)).astype("int8"),
    )
    return df

analytic_pressor = add_inhospital_death_features(analytic_pressor).persist()
analytic_vent = add_inhospital_death_features(analytic_vent).persist()

# Sanity checks
print("pressor columns containing 'hospital_expire':",
      [c for c in analytic_pressor.columns if "hospital_expire" in c])

print("vent columns containing 'hospital_expire':",
      [c for c in analytic_vent.columns if "hospital_expire" in c])

print("Death event in hosp after trigger rate (pressor):", analytic_pressor["death_event_in_hosp"].mean().compute())
print("Death event in hosp after trigger rate (vent):", analytic_vent["death_event_in_hosp"].mean().compute())

print("Mismatch: hospital_expire_flag==1 but deathtime missing (pressor):",
      ((analytic_pressor["hospital_expire_flag"] == 1) & (analytic_pressor["deathtime"].isna())).mean().compute())

print("Mismatch: hospital_expire_flag==1 but deathtime missing (vent):",
      ((analytic_vent["hospital_expire_flag"] == 1) & (analytic_vent["deathtime"].isna())).mean().compute())


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+------------------------------+------------+-------------+
| Merge columns                | left dtype | right dtype |
+------------------------------+------------+-------------+
| ('subject_id', 'subject_id') | Int64      | float64     |
| ('hadm_id', 'hadm_id')       | Int64      | float64     |
+------------------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+------------------------------+------------+-------------+
| Merge columns                | left dtype | right dtype |
+------------------------------+------------+-------------+
| ('subject_id', 'subject_i

pressor columns containing 'hospital_expire': ['hospital_expire_flag']
vent columns containing 'hospital_expire': ['hospital_expire_flag']
Death event in hosp after trigger rate (pressor): 0.13453620906346156
Death event in hosp after trigger rate (vent): 0.2122378238646378
Mismatch: hospital_expire_flag==1 but deathtime missing (pressor): 0.00013001459052627018
Mismatch: hospital_expire_flag==1 but deathtime missing (vent): 8.812643205452088e-05


In [28]:
import dask.dataframe as dd

triage_path = TARGET_DIRS["ed"] / "triage.csv.gz"
edstays_path = TARGET_DIRS["ed"] / "edstays.csv.gz"

#  only load required columns to avoid dtype inference issues in unrelated fields (e.g., pain)
req_triage = ["stay_id", "chiefcomplaint"]
req_edstays = ["subject_id", "hadm_id", "stay_id", "intime", "outtime"]

triage = dd.read_csv(
    triage_path,
    compression="gzip",
    usecols=req_triage,
    assume_missing=True,
    blocksize=None,
)

edstays = dd.read_csv(
    edstays_path,
    compression="gzip",
    usecols=req_edstays,
    assume_missing=True,
    blocksize=None,
)

#  parse timestamps
edstays["intime"] = dd.to_datetime(edstays["intime"])
edstays["outtime"] = dd.to_datetime(edstays["outtime"])

#  keep ED stays linked to hospital admission
ed_linked = edstays[edstays["hadm_id"].notnull()]

#  pick the last ED stay per (subject_id, hadm_id) using max(outtime)
max_out = ed_linked.groupby(["subject_id", "hadm_id"])["outtime"].max().reset_index()
ed_last = ed_linked.merge(max_out, on=["subject_id", "hadm_id", "outtime"], how="inner")

#  add chiefcomplaint
ed_text = ed_last.merge(triage, on="stay_id", how="left")

#  keep minimal ED text fields and avoid name collisions
ed_text = ed_text.rename(columns={"stay_id": "ed_stay_id", "intime": "ed_intime", "outtime": "ed_outtime"})
ed_text = ed_text[["subject_id", "hadm_id", "ed_stay_id", "ed_intime", "ed_outtime", "chiefcomplaint"]]

#  attach to analytic tables
analytic_pressor = analytic_pressor.merge(ed_text, on=["subject_id", "hadm_id"], how="left").persist()
analytic_vent = analytic_vent.merge(ed_text, on=["subject_id", "hadm_id"], how="left").persist()

print("ED chiefcomplaint attached.")
print("pressor chiefcomplaint missing rate:", analytic_pressor["chiefcomplaint"].isnull().mean().compute())
print("vent chiefcomplaint missing rate:", analytic_vent["chiefcomplaint"].isnull().mean().compute())


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+------------------------------+------------+-------------+
| Merge columns                | left dtype | right dtype |
+------------------------------+------------+-------------+
| ('subject_id', 'subject_id') | Int64      | float64     |
| ('hadm_id', 'hadm_id')       | Int64      | float64     |
+------------------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+------------------------------+------------+-------------+
| Merge columns                | left dtype | right dtype |
+------------------------------+------------+-------------+
| ('subject_id', 'subject_i

ED chiefcomplaint attached.
pressor chiefcomplaint missing rate: 0.6357569016078471
vent chiefcomplaint missing rate: 0.6257564185418013


In [29]:
# %%
import numpy as np

# ---- Pressor dataset schema fixes ----
analytic_pressor = analytic_pressor.assign(
    trigger_value=analytic_pressor["trigger_value"].astype("float64"),
    time_since_icu_admission_hours=analytic_pressor["time_since_icu_admission_hours"].astype("float64"),
    time_to_pressor_hours=analytic_pressor["time_to_pressor_hours"].astype("float64"),
    surv_tau_hours=analytic_pressor["surv_tau_hours"].astype("float64"),
    surv_event_time_hours=analytic_pressor["surv_event_time_hours"].astype("float64"),
    surv_icu_follow_up_hours=analytic_pressor["surv_icu_follow_up_hours"].astype("float64"),
    surv_follow_up_hours=analytic_pressor["surv_follow_up_hours"].astype("float64"),
    surv_duration_hours=analytic_pressor["surv_duration_hours"].astype("float64"),
)

num_cols_pressor = [c for c in analytic_pressor.columns if c.endswith(("_last", "_mean", "_min", "_max"))]
analytic_pressor = analytic_pressor.astype({c: "float64" for c in num_cols_pressor})

flag_cols_pressor = [
    "elderly", "hospital_expire_flag", "pressor_started_within_2h",
    "pressor_active_at_t0", "pressor_initiated_within_2h",
    "surv_in_risk", "surv_event", "surv_censored",
    "surv_censor_admin_tau", "surv_censor_icu_discharge",
]
analytic_pressor = analytic_pressor.astype({c: "int8" for c in flag_cols_pressor})
analytic_pressor = analytic_pressor.persist()

# ---- Vent dataset schema fixes ----
analytic_vent = analytic_vent.assign(
    trigger_value=analytic_vent["trigger_value"].astype("float64"),
    time_since_icu_admission_hours=analytic_vent["time_since_icu_admission_hours"].astype("float64"),
    time_to_vent_hours=analytic_vent["time_to_vent_hours"].astype("float64"),
    surv_tau_hours=analytic_vent["surv_tau_hours"].astype("float64"),
    surv_event_time_hours=analytic_vent["surv_event_time_hours"].astype("float64"),
    surv_icu_follow_up_hours=analytic_vent["surv_icu_follow_up_hours"].astype("float64"),
    surv_follow_up_hours=analytic_vent["surv_follow_up_hours"].astype("float64"),
    surv_duration_hours=analytic_vent["surv_duration_hours"].astype("float64"),
)

num_cols_vent = [c for c in analytic_vent.columns if c.endswith(("_last", "_mean", "_min", "_max"))]
analytic_vent = analytic_vent.astype({c: "float64" for c in num_cols_vent})

flag_cols_vent = [
    "elderly", "hospital_expire_flag", "vent_started_within_6h",
    "vent_active_at_t0", "vent_initiated_within_6h",
    "surv_in_risk", "surv_event", "surv_censored",
    "surv_censor_admin_tau", "surv_censor_icu_discharge",
]
analytic_vent = analytic_vent.astype({c: "int8" for c in flag_cols_vent})
analytic_vent = analytic_vent.persist()

# ---- Sanity checks (dtypes that matter) ----
print("pressor trigger_value dtype:", analytic_pressor["trigger_value"].dtype)
print("pressor surv_duration_hours dtype:", analytic_pressor["surv_duration_hours"].dtype)
print("vent trigger_value dtype:", analytic_vent["trigger_value"].dtype)
print("vent surv_duration_hours dtype:", analytic_vent["surv_duration_hours"].dtype)

# ---- Write to current directory ----
analytic_pressor.to_parquet("analytic_pressor_trigger_MAP_lt_65.parquet", write_index=False)
analytic_vent.to_parquet("analytic_vent_trigger_SpO2_lt_90.parquet", write_index=False)

print("Saved:")
print(" - analytic_pressor_trigger_MAP_lt_65.parquet")
print(" - analytic_vent_trigger_SpO2_lt_90.parquet")


pressor trigger_value dtype: float64
pressor surv_duration_hours dtype: float64
vent trigger_value dtype: float64
vent surv_duration_hours dtype: float64
Saved:
 - analytic_pressor_trigger_MAP_lt_65.parquet
 - analytic_vent_trigger_SpO2_lt_90.parquet


In [30]:
import pandas as pd

# -----------------------------
# 1) Trigger must be within ICU window
# -----------------------------
def check_trigger_outside_icu(triggers_dd, name):
    df = triggers_dd[["stay_id", "intime", "outtime", "trigger_time"]].compute()
    df["trigger_outside_icu"] = (df["trigger_time"] < df["intime"]) | (df["trigger_time"] > df["outtime"])
    rate = df["trigger_outside_icu"].mean()
    print(f"[{name}] trigger_outside_icu rate = {rate:.6f} (n={len(df)})")
    if rate > 0:
        display(df[df["trigger_outside_icu"]].head(20))

check_trigger_outside_icu(map_triggers,  "MAP < 65")
check_trigger_outside_icu(spo2_triggers, "SpO2 < 90")


# -----------------------------
# 2) Treatment event time must not exceed ICU outtime
#    Use map_labeled/spo2_labeled because they contain *_starttime columns.
# -----------------------------
def check_event_after_outtime(labeled_dd, start_col, name):
    df = labeled_dd[["stay_id", "outtime", "trigger_time", start_col]].compute()
    df = df[df[start_col].notna()]
    if len(df) == 0:
        print(f"[{name}] no events, skip")
        return
    df["event_after_outtime"] = df[start_col] > df["outtime"]
    rate = df["event_after_outtime"].mean()
    print(f"[{name}] event_after_outtime rate = {rate:.6f} (n_events={len(df)})")
    if rate > 0:
        display(df[df["event_after_outtime"]].head(20))

check_event_after_outtime(map_labeled,  start_col="pressor_starttime", name="pressor")
check_event_after_outtime(spo2_labeled, start_col="vent_starttime",    name="vent")


# -----------------------------
# 3) Treatment event time must be strictly after trigger_time by your definition
# -----------------------------
def check_event_before_or_at_trigger(labeled_dd, start_col, name):
    df = labeled_dd[["stay_id", "trigger_time", start_col]].compute()
    df = df[df[start_col].notna()]
    if len(df) == 0:
        print(f"[{name}] no events, skip")
        return
    df["event_before_or_at_trigger"] = df[start_col] <= df["trigger_time"]
    rate = df["event_before_or_at_trigger"].mean()
    print(f"[{name}] event_before_or_at_trigger rate = {rate:.6f} (n_events={len(df)})")
    if rate > 0:
        display(df[df["event_before_or_at_trigger"]].head(20))

check_event_before_or_at_trigger(map_labeled,  start_col="pressor_starttime", name="pressor")
check_event_before_or_at_trigger(spo2_labeled, start_col="vent_starttime",    name="vent")


# -----------------------------
# 4) Flag/time_to consistency checks (validity check on labeling logic)
# -----------------------------
def check_flag_time_consistency(analytic_dd, flag_col, time_col, window_hours, name):
    df = analytic_dd[["stay_id", flag_col, time_col]].compute()

    #  when flagged True, time_to should be (0, window] (and not null)
    flagged = df[df[flag_col] == True]
    if len(flagged) > 0:
        bad_time = ((flagged[time_col].isna()) | (flagged[time_col] <= 0) | (flagged[time_col] > window_hours)).mean()
        print(f"[{name}] flagged rows bad {time_col} rate = {bad_time:.6f} (n_flagged={len(flagged)})")
        if bad_time > 0:
            display(flagged[(flagged[time_col].isna()) | (flagged[time_col] <= 0) | (flagged[time_col] > window_hours)].head(20))
    else:
        print(f"[{name}] no flagged True rows")

    #  when flagged False, time_to should be null
    not_flagged = df[df[flag_col] == False]
    if len(not_flagged) > 0:
        nonnull_time = not_flagged[time_col].notna().mean()
        print(f"[{name}] unflagged rows non-null {time_col} rate = {nonnull_time:.6f} (n_unflagged={len(not_flagged)})")
        if nonnull_time > 0:
            display(not_flagged[not_flagged[time_col].notna()].head(20))

check_flag_time_consistency(
    analytic_pressor,
    flag_col="pressor_started_within_2h",
    time_col="time_to_pressor_hours",
    window_hours=2,
    name="pressor"
)

check_flag_time_consistency(
    analytic_vent,
    flag_col="vent_started_within_6h",
    time_col="time_to_vent_hours",
    window_hours=6,
    name="vent"
)


[MAP < 65] trigger_outside_icu rate = 0.000000 (n=69223)
[SpO2 < 90] trigger_outside_icu rate = 0.000000 (n=34042)
[pressor] event_after_outtime rate = 0.000000 (n_events=15242)
[vent] event_after_outtime rate = 0.000000 (n_events=1667)
[pressor] event_before_or_at_trigger rate = 0.000000 (n_events=15242)
[vent] event_before_or_at_trigger rate = 0.000000 (n_events=1667)
[pressor] flagged rows bad time_to_pressor_hours rate = 0.000000 (n_flagged=15242)
[pressor] unflagged rows non-null time_to_pressor_hours rate = 0.000000 (n_unflagged=53981)
[vent] flagged rows bad time_to_vent_hours rate = 0.000000 (n_flagged=1667)
[vent] unflagged rows non-null time_to_vent_hours rate = 0.000000 (n_unflagged=32375)


## Data Cleaning (Radiology)

In [7]:
# ED-indexed radiology decision-family dataset + ED diagnosis + radiology report text (0-6h)
# FULL FIX (end-to-end) for YOUR ACTUAL radiology_detail schema:
# radiology_detail columns: ['note_id','field_name','field_value']
# field_name universe (observed): addendum_note_id, cpt_code, exam_code, exam_name, parent_note_id
#
# Outputs:
# - one row per ED stay (ed_stay_id)
# - index_order_subtype, index_poe_id, index_rad_ordertime at ED-stay level
# - radiology report text restricted to [first_rad_order_time, ed_end]
# - exam_name/exam_code/cpt_code/parent_note_id/addendum_note_id merged from EAV pivot
# - vitalsign summaries pre-first_rad_order_time (pain fixed)
# - fixed named lab last-values pre-first_rad_order_time
# - parquet dtype normalization for pyarrow

import dask.dataframe as dd
import pandas as pd
import numpy as np

MAX_H = 6.0
windows = [2.0, 4.0, 6.0]
LAB_ITEMIDS = {
    "lactate":    [50813, 52442, 53154],
    "creatinine": [50912, 52546],
    "wbc":        [51301, 51755, 51756],
    "platelets":  [51265, 53189],
    "sodium":     [50983, 52623],
    "potassium":  [50971, 52610],
}

triage_path        = TARGET_DIRS["ed"]   / "triage.csv.gz"
edstays_path       = TARGET_DIRS["ed"]   / "edstays.csv.gz"
diagnosis_path     = TARGET_DIRS["ed"]   / "diagnosis.csv.gz"
vitalsign_path     = TARGET_DIRS["ed"]   / "vitalsign.csv.gz"

poe_path           = TARGET_DIRS["hosp"] / "poe.csv.gz"
poe_detail_path    = TARGET_DIRS["hosp"] / "poe_detail.csv.gz"
labevents_path     = TARGET_DIRS["hosp"] / "labevents.csv.gz"

patients_path      = TARGET_DIRS["hosp"] / "patients.csv.gz"
admissions_path    = TARGET_DIRS["hosp"] / "admissions.csv.gz"

note_radiology_path        = TARGET_DIRS["notes"] / "radiology.csv.gz"
note_radiology_detail_path = TARGET_DIRS["notes"] / "radiology_detail.csv.gz"

# ----------------------------
# Read core tables
# ----------------------------
triage = dd.read_csv(
    triage_path, compression="gzip", assume_missing=True, blocksize=None, dtype={"pain": "object"}
)
edstays = dd.read_csv(edstays_path, compression="gzip", assume_missing=True, blocksize=None)
dx = dd.read_csv(
    diagnosis_path, compression="gzip", assume_missing=True, blocksize=None,
    dtype={"icd_code": "object", "icd_title": "object"}
)

poe = dd.read_csv(
    poe_path, compression="gzip", assume_missing=True, blocksize=None,
    dtype={"discontinued_by_poe_id": "object", "poe_id": "object"}
)
poe_detail = dd.read_csv(
    poe_detail_path, compression="gzip", assume_missing=True, blocksize=None,
    dtype={"poe_id": "object", "field_name": "object", "field_value": "object"}
)

patients = dd.read_csv(patients_path, compression="gzip", assume_missing=True, blocksize=None)
admissions = dd.read_csv(
    admissions_path, compression="gzip", assume_missing=True, blocksize=None,
    usecols=["subject_id", "hadm_id", "insurance", "language", "deathtime"],
    dtype={"insurance": "object", "language": "object", "deathtime": "object"}
)
admissions["deathtime"] = dd.to_datetime(admissions["deathtime"], errors="coerce")

vitalsign = dd.read_csv(
    vitalsign_path,
    compression="gzip",
    assume_missing=True,
    blocksize=None,
    dtype={"pain": "object", "rhythm": "object"},
)
vitalsign["charttime"] = dd.to_datetime(vitalsign["charttime"], errors="coerce")
vitalsign["pain_num"] = dd.to_numeric(vitalsign["pain"], errors="coerce")

labevents = dd.read_csv(
    labevents_path,
    compression="gzip",
    assume_missing=True,
    blocksize=None,
    usecols=["subject_id", "hadm_id", "charttime", "itemid", "valuenum"],
    dtype={"itemid": "object", "valuenum": "object"},
)
labevents["charttime"] = dd.to_datetime(labevents["charttime"], errors="coerce")
labevents["itemid"] = dd.to_numeric(labevents["itemid"], errors="coerce")
labevents["valuenum"] = dd.to_numeric(labevents["valuenum"], errors="coerce")

edstays["intime"] = dd.to_datetime(edstays["intime"])
edstays["outtime"] = dd.to_datetime(edstays["outtime"])
poe["ordertime"] = dd.to_datetime(poe["ordertime"])

# ----------------------------
# ED base (admitted ED stays) + triage
# ----------------------------
req_triage = [
    "stay_id", "temperature", "heartrate", "resprate", "o2sat",
    "sbp", "dbp", "pain", "acuity", "chiefcomplaint"
]
req_edstays = [
    "subject_id", "hadm_id", "stay_id", "intime", "outtime",
    "gender", "race", "arrival_transport", "disposition"
]

triage = triage[req_triage]
edstays = edstays[req_edstays]

patients = patients[["subject_id", "anchor_age"]]
admissions = admissions[["subject_id", "hadm_id", "insurance", "language", "deathtime"]]

ed_base = edstays[edstays["hadm_id"].notnull()].merge(triage, on="stay_id", how="left")
ed_base = ed_base.rename(columns={"stay_id": "ed_stay_id", "intime": "ed_intime", "outtime": "ed_outtime"})

cutoff_delta = pd.to_timedelta(MAX_H, unit="h")
ed_base["ed_cutoff"] = ed_base["ed_intime"] + cutoff_delta
ed_base["ed_end"] = ed_base["ed_outtime"].where(ed_base["ed_outtime"] < ed_base["ed_cutoff"], ed_base["ed_cutoff"])

ed_base = ed_base.merge(admissions, on=["subject_id", "hadm_id"], how="left")
ed_base = ed_base.merge(patients, on=["subject_id"], how="left").persist()

# ----------------------------
# ED diagnosis string
# ----------------------------
dx = dx[["subject_id", "stay_id", "seq_num", "icd_code", "icd_version", "icd_title"]]
dx_link = ed_base[["subject_id", "ed_stay_id"]].merge(
    dx, left_on=["subject_id", "ed_stay_id"], right_on=["subject_id", "stay_id"], how="left"
)

dx_pd = dx_link[["subject_id", "ed_stay_id", "seq_num", "icd_code", "icd_version", "icd_title"]].compute()
dx_pd = dx_pd.sort_values(["subject_id", "ed_stay_id", "seq_num"])

def fmt_dx_row(r):
    if isinstance(r["icd_code"], str):
        if isinstance(r["icd_title"], str):
            return f'{r["icd_code"]} ({r["icd_title"]})'
        return r["icd_code"]
    return None

dx_pd["dx_item"] = dx_pd.apply(fmt_dx_row, axis=1)

dx_agg_pd = (
    dx_pd.groupby(["subject_id", "ed_stay_id"])["dx_item"]
    .apply(lambda s: " | ".join([x for x in s.tolist() if isinstance(x, str)]))
    .reset_index()
    .rename(columns={"dx_item": "ed_dx_str"})
)
dx_agg_dd = dd.from_pandas(dx_agg_pd, npartitions=max(1, ed_base.npartitions // 4))
ed_base = ed_base.merge(dx_agg_dd, on=["subject_id", "ed_stay_id"], how="left").persist()

# ----------------------------
# Radiology orders within ED window
# ----------------------------
rad_orders = poe[poe["order_type"] == "Radiology"][["subject_id", "hadm_id", "ordertime", "order_subtype", "poe_id"]]

order_level = ed_base[["subject_id", "hadm_id", "ed_stay_id", "ed_intime", "ed_end"]].merge(
    rad_orders, on=["subject_id", "hadm_id"], how="left"
)
order_level["dt_hours_from_ed"] = (order_level["ordertime"] - order_level["ed_intime"]).dt.total_seconds() / 3600.0
order_level = order_level[
    (order_level["ordertime"] >= order_level["ed_intime"]) &
    (order_level["ordertime"] <= order_level["ed_end"])
].persist()

is_ct   = (order_level["order_subtype"] == "CT Scan")
is_mri  = (order_level["order_subtype"] == "MRI")
is_xray = (order_level["order_subtype"] == "General Xray")
is_adv  = is_ct | is_mri

def min_time_dd(df, name):
    return (
        df.groupby(["subject_id", "hadm_id", "ed_stay_id"])["dt_hours_from_ed"]
        .min()
        .reset_index()
        .rename(columns={"dt_hours_from_ed": name})
    )

def count_within_dd(df, h, name):
    t = df[df["dt_hours_from_ed"] <= h]
    return (
        t.groupby(["subject_id", "hadm_id", "ed_stay_id"])["poe_id"]
        .count()
        .reset_index()
        .rename(columns={"poe_id": name})
    )

t_any = min_time_dd(order_level[order_level["poe_id"].notnull()], "time_to_any_rad_hours")
t_adv = min_time_dd(order_level[is_adv], "time_to_advanced_hours")
t_xr  = min_time_dd(order_level[is_xray], "time_to_xray_hours")

analytic_rad = ed_base.merge(t_any, on=["subject_id", "hadm_id", "ed_stay_id"], how="left")
analytic_rad = analytic_rad.merge(t_adv, on=["subject_id", "hadm_id", "ed_stay_id"], how="left")
analytic_rad = analytic_rad.merge(t_xr,  on=["subject_id", "hadm_id", "ed_stay_id"], how="left")

analytic_rad["first_rad_order_time"] = analytic_rad["ed_intime"] + dd.to_timedelta(
    analytic_rad["time_to_any_rad_hours"], unit="h"
)

for h in windows:
    hh = int(h)
    cnt_any = count_within_dd(order_level[order_level["poe_id"].notnull()], h, f"n_rad_orders_{hh}h")
    cnt_adv = count_within_dd(order_level[is_adv], h, f"n_advanced_orders_{hh}h")
    cnt_xr  = count_within_dd(order_level[is_xray], h, f"n_xray_orders_{hh}h")

    analytic_rad = analytic_rad.merge(cnt_any, on=["subject_id", "hadm_id", "ed_stay_id"], how="left")
    analytic_rad = analytic_rad.merge(cnt_adv, on=["subject_id", "hadm_id", "ed_stay_id"], how="left")
    analytic_rad = analytic_rad.merge(cnt_xr,  on=["subject_id", "hadm_id", "ed_stay_id"], how="left")

for h in windows:
    hh = int(h)
    analytic_rad[f"rad_any_within_{hh}h"] = (
        (analytic_rad["time_to_any_rad_hours"].notnull()) & (analytic_rad["time_to_any_rad_hours"] <= h)
    ).astype("int8")
    analytic_rad[f"rad_advanced_within_{hh}h"] = (
        (analytic_rad["time_to_advanced_hours"].notnull()) & (analytic_rad["time_to_advanced_hours"] <= h)
    ).astype("int8")
    analytic_rad[f"rad_xray_within_{hh}h"] = (
        (analytic_rad["time_to_xray_hours"].notnull()) & (analytic_rad["time_to_xray_hours"] <= h)
    ).astype("int8")

analytic_rad = analytic_rad.assign(trigger_type="ED_intime", trigger_time=analytic_rad["ed_intime"]).persist()

# ----------------------------
# Index radiology order row -> subtype + poe_id + ordertime at stay level
# ----------------------------
idx_order_pd = (
    order_level.loc[order_level["poe_id"].notnull(),
                    ["subject_id", "hadm_id", "ed_stay_id", "ordertime", "order_subtype", "poe_id"]]
    .compute()
    .sort_values(["subject_id", "hadm_id", "ed_stay_id", "ordertime"])
)
idx_order_first_pd = (
    idx_order_pd.groupby(["subject_id", "hadm_id", "ed_stay_id"], as_index=False)
    .head(1)
    .rename(columns={
        "ordertime": "index_rad_ordertime",
        "order_subtype": "index_order_subtype",
        "poe_id": "index_poe_id",
    })
)
idx_order_first_dd = dd.from_pandas(idx_order_first_pd, npartitions=max(1, ed_base.npartitions // 4))
analytic_rad = analytic_rad.merge(idx_order_first_dd, on=["subject_id", "hadm_id", "ed_stay_id"], how="left").persist()

# ----------------------------
# Pre-index-order vitalsign summaries
# ----------------------------
vs_cols = ["subject_id","stay_id","charttime","temperature","heartrate","resprate","o2sat","sbp","dbp","pain_num"]
vitalsign_small = vitalsign[vs_cols]

vs_link = analytic_rad[["subject_id","ed_stay_id","ed_intime","first_rad_order_time"]].merge(
    vitalsign_small,
    left_on=["subject_id","ed_stay_id"],
    right_on=["subject_id","stay_id"],
    how="left",
)

vs_link = vs_link[
    (vs_link["first_rad_order_time"].notnull()) &
    (vs_link["charttime"] >= vs_link["ed_intime"]) &
    (vs_link["charttime"] <= vs_link["first_rad_order_time"])
].persist()

vs_agg_spec = {c: ["min", "max"] for c in ["temperature","heartrate","resprate","o2sat","sbp","dbp","pain_num"]}
vs_agg_spec["charttime"] = "count"

vs_agg = vs_link.groupby(["subject_id","ed_stay_id"]).agg(vs_agg_spec).reset_index()
vs_agg.columns = ["_".join([x for x in col if x]) if isinstance(col, tuple) else col for col in vs_agg.columns]
vs_agg = vs_agg.rename(columns={
    "charttime_count": "n_vitalsign_rows_pre_order",
    "pain_num_min": "pain_min_pre_order",
    "pain_num_max": "pain_max_pre_order",
})

def last_value_dd(df, value_col, out_col):
    sub = df[df[value_col].notnull()][["subject_id", "ed_stay_id", "charttime", value_col]]
    last_t = (
        sub.groupby(["subject_id", "ed_stay_id"])["charttime"]
        .max()
        .reset_index()
        .rename(columns={"charttime": "last_charttime"})
    )
    last_rows = sub.merge(last_t, on=["subject_id", "ed_stay_id"], how="inner")
    last_rows = last_rows[last_rows["charttime"] == last_rows["last_charttime"]]
    out = (
        last_rows.groupby(["subject_id", "ed_stay_id"])[value_col]
        .mean()
        .reset_index()
        .rename(columns={value_col: out_col})
    )
    return out

vs_last_specs = [
    ("temperature", "temp_last"),
    ("heartrate", "hr_last"),
    ("resprate", "rr_last"),
    ("o2sat", "spo2_last"),
    ("sbp", "sbp_last"),
    ("dbp", "dbp_last"),
    ("pain_num", "pain_last"),
]

for value_col, out_col in vs_last_specs:
    analytic_rad = analytic_rad.merge(
        last_value_dd(vs_link, value_col, out_col),
        on=["subject_id", "ed_stay_id"],
        how="left",
    )

analytic_rad = analytic_rad.merge(vs_agg, on=["subject_id","ed_stay_id"], how="left").persist()
analytic_rad["map_last"] = (analytic_rad["sbp_last"] + 2.0 * analytic_rad["dbp_last"]) / 3.0

# ----------------------------
# Pre-index-order labs summaries (fixed ICU-style lab names; last value before first order)
# ----------------------------
lab_map_pd = []
for lab_name, itemids in LAB_ITEMIDS.items():
    for itemid in itemids:
        lab_map_pd.append({"itemid": itemid, "lab_name": lab_name})
lab_map_pd = pd.DataFrame(lab_map_pd)
lab_map_dd = dd.from_pandas(lab_map_pd, npartitions=1)

all_lab_itemids = lab_map_pd["itemid"].tolist()

labs_named = labevents[["subject_id", "hadm_id", "charttime", "itemid", "valuenum"]]
labs_named = labs_named[labs_named["itemid"].isin(all_lab_itemids)]
labs_named = labs_named.merge(lab_map_dd, on="itemid", how="inner")

labs_link = analytic_rad[["subject_id", "hadm_id", "ed_stay_id", "ed_intime", "first_rad_order_time"]].merge(
    labs_named,
    on=["subject_id", "hadm_id"],
    how="left",
)

labs_link = labs_link[
    (labs_link["first_rad_order_time"].notnull()) &
    (labs_link["charttime"] >= labs_link["ed_intime"]) &
    (labs_link["charttime"] <= labs_link["first_rad_order_time"])
].persist()

labs_nonnull = labs_link[labs_link["valuenum"].notnull()][["ed_stay_id", "lab_name", "charttime", "valuenum"]]

labs_last_t = (
    labs_nonnull.groupby(["ed_stay_id", "lab_name"])["charttime"]
    .max()
    .reset_index()
    .rename(columns={"charttime": "last_charttime"})
)

labs_last_rows = labs_nonnull.merge(labs_last_t, on=["ed_stay_id", "lab_name"], how="inner")
labs_last_rows = labs_last_rows[labs_last_rows["charttime"] == labs_last_rows["last_charttime"]]

labs_last = (
    labs_last_rows.groupby(["ed_stay_id", "lab_name"])["valuenum"]
    .mean()
    .reset_index()
    .rename(columns={"valuenum": "lab_last"})
)

labs_last_pd = labs_last.compute()
labs_last_wide_pd = labs_last_pd.pivot(index="ed_stay_id", columns="lab_name", values="lab_last").reset_index()
labs_last_wide_pd = labs_last_wide_pd.rename(columns={
    "lactate": "lactate_last",
    "creatinine": "creatinine_last",
    "wbc": "wbc_last",
    "platelets": "platelets_last",
    "sodium": "sodium_last",
    "potassium": "potassium_last",
})

labs_last_wide_dd = dd.from_pandas(labs_last_wide_pd, npartitions=max(1, analytic_rad.npartitions // 2))
analytic_rad = analytic_rad.merge(labs_last_wide_dd, on="ed_stay_id", how="left").persist()

# ----------------------------
# POE_DETAIL enrichment (order-level)
# ----------------------------
poe_ids_pd = order_level[order_level["poe_id"].notnull()][["poe_id"]].drop_duplicates().compute()
poe_detail_sub = poe_detail.merge(dd.from_pandas(poe_ids_pd, npartitions=1), on="poe_id", how="inner")[
    ["poe_id", "field_name", "field_value"]
]

poe_detail_pd = poe_detail_sub.compute().sort_values(["poe_id", "field_name"])
poe_detail_agg_pd = (
    poe_detail_pd.groupby("poe_id")[["field_name", "field_value"]]
    .apply(lambda df: " | ".join([
        f"{fn}: {fv}" for fn, fv in zip(df["field_name"].tolist(), df["field_value"].tolist())
        if isinstance(fn, str) and isinstance(fv, str)
    ]))
    .reset_index()
    .rename(columns={0: "poe_detail_str"})
)
poe_detail_agg_dd = dd.from_pandas(poe_detail_agg_pd, npartitions=max(1, order_level.npartitions // 2))
order_level_enriched = order_level.merge(poe_detail_agg_dd, on="poe_id", how="left").persist()

# ----------------------------
# Radiology notes + radiology_detail(EAV) pivot
# ----------------------------
rad_notes = dd.read_csv(
    note_radiology_path,
    compression="gzip",
    assume_missing=True,
    blocksize=None,
    dtype={"note_id": "object", "text": "object"},
)
rad_notes["charttime"] = dd.to_datetime(rad_notes["charttime"], errors="coerce")
rad_notes_small = rad_notes[["note_id", "subject_id", "hadm_id", "charttime", "text"]]

rad_detail = dd.read_csv(
    note_radiology_detail_path,
    compression="gzip",
    assume_missing=True,
    blocksize=None,
    dtype={"note_id": "object", "field_name": "object", "field_value": "object"},
)[["note_id", "field_name", "field_value"]]

# HARD ASSERTS: crash if file content differs from what you observed
observed_fields = rad_detail["field_name"].value_counts().compute().index.tolist()
expected_fields = ["addendum_note_id", "cpt_code", "exam_code", "exam_name", "parent_note_id"]
missing_fields = [f for f in expected_fields if f not in observed_fields]
assert len(missing_fields) == 0, f"radiology_detail missing fields {missing_fields}; observed={observed_fields}"

rad_detail = rad_detail[rad_detail["field_name"].isin(expected_fields)]

# Restrict to note_ids present in radiology notes to keep pivot sane
note_ids_pd = rad_notes_small[["note_id"]].drop_duplicates().compute()
rad_detail_sub = rad_detail.merge(dd.from_pandas(note_ids_pd, npartitions=1), on="note_id", how="inner")
rad_detail_pd = rad_detail_sub.compute()

rad_detail_wide_pd = (
    rad_detail_pd.pivot_table(
        index="note_id",
        columns="field_name",
        values="field_value",
        aggfunc="first",
    )
    .reset_index()
)

rad_detail_wide_dd = dd.from_pandas(rad_detail_wide_pd, npartitions=max(1, rad_notes_small.npartitions // 4))
rad_notes_w_detail = rad_notes_small.merge(rad_detail_wide_dd, on="note_id", how="left")

# ----------------------------
# Align reports to first radiology order time: [first_rad_order_time, ed_end]
# ----------------------------
notes_link = analytic_rad[["subject_id","hadm_id","ed_stay_id","ed_intime","ed_end","first_rad_order_time"]].merge(
    rad_notes_w_detail,
    on=["subject_id","hadm_id"],
    how="left",
)

notes_link = notes_link[
    (notes_link["first_rad_order_time"].notnull()) &
    (notes_link["charttime"] >= notes_link["first_rad_order_time"]) &
    (notes_link["charttime"] <= notes_link["ed_end"])
].persist()

notes_link["dt_hours_report_from_ed"] = (notes_link["charttime"] - notes_link["ed_intime"]).dt.total_seconds() / 3600.0
note_level_out = notes_link.persist()

# ----------------------------
# Aggregate radiology text + detail fields to ED-stay level
# ----------------------------
cols_for_pd = [
    "subject_id", "hadm_id", "ed_stay_id", "charttime", "note_id", "text", "dt_hours_report_from_ed",
    "exam_name", "exam_code", "cpt_code", "parent_note_id", "addendum_note_id",
]
notes_pd = note_level_out[cols_for_pd].compute()
notes_pd = notes_pd.sort_values(["ed_stay_id", "charttime", "note_id"])

txt_all_pd = (
    notes_pd.groupby(["subject_id", "hadm_id", "ed_stay_id"])["text"]
    .apply(lambda s: "\n\n".join([t for t in s.tolist() if isinstance(t, str)]))
    .reset_index()
    .rename(columns={"text": "rad_txt_0_6h_all"})
)

idx = notes_pd.groupby(["subject_id","hadm_id","ed_stay_id"])["charttime"].idxmin()
first_pd = notes_pd.loc[idx, :].copy()
first_pd = first_pd.rename(columns={
    "charttime": "first_report_time",
    "note_id": "first_note_id",
    "text": "rad_txt_first",
    "dt_hours_report_from_ed": "first_report_dt_hours",
})

keep_first_cols = [
    "subject_id","hadm_id","ed_stay_id",
    "first_report_time","first_note_id","rad_txt_first","first_report_dt_hours",
    "exam_name","exam_code","cpt_code","parent_note_id","addendum_note_id",
]
first_pd = first_pd[keep_first_cols]

txt_all_dd = dd.from_pandas(txt_all_pd, npartitions=max(1, analytic_rad.npartitions // 2))
first_dd   = dd.from_pandas(first_pd,   npartitions=max(1, analytic_rad.npartitions // 2))

analytic_rad_w_text = analytic_rad.merge(txt_all_dd, on=["subject_id","hadm_id","ed_stay_id"], how="left")
analytic_rad_w_text = analytic_rad_w_text.merge(first_dd,   on=["subject_id","hadm_id","ed_stay_id"], how="left").persist()

# ----------------------------
# Follow-up radiology orders AFTER first report time, within ED window
# ----------------------------
follow = analytic_rad_w_text[["subject_id","hadm_id","ed_stay_id","first_report_time","ed_end"]].merge(
    rad_orders, on=["subject_id","hadm_id"], how="left"
)
follow = follow[
    (follow["first_report_time"].notnull()) &
    (follow["ordertime"] > follow["first_report_time"]) &
    (follow["ordertime"] <= follow["ed_end"])
].persist()

follow["dt_hours_after_report"] = (follow["ordertime"] - follow["first_report_time"]).dt.total_seconds() / 3600.0

is_ct_f   = (follow["order_subtype"] == "CT Scan")
is_mri_f  = (follow["order_subtype"] == "MRI")
is_xray_f = (follow["order_subtype"] == "General Xray")
is_adv_f  = is_ct_f | is_mri_f

def follow_count(df, h, name):
    t = df[df["dt_hours_after_report"] <= h]
    return (
        t.groupby(["subject_id","hadm_id","ed_stay_id"])["poe_id"]
        .count()
        .reset_index()
        .rename(columns={"poe_id": name})
    )

def follow_min(df, name):
    return (
        df.groupby(["subject_id","hadm_id","ed_stay_id"])["dt_hours_after_report"]
        .min()
        .reset_index()
        .rename(columns={"dt_hours_after_report": name})
    )

f_any_t = follow_min(follow[follow["poe_id"].notnull()], "time_to_followup_any_hours")
f_adv_t = follow_min(follow[is_adv_f], "time_to_followup_advanced_hours")
f_xr_t  = follow_min(follow[is_xray_f], "time_to_followup_xray_hours")

analytic_rad_w_text = analytic_rad_w_text.merge(f_any_t, on=["subject_id","hadm_id","ed_stay_id"], how="left")
analytic_rad_w_text = analytic_rad_w_text.merge(f_adv_t, on=["subject_id","hadm_id","ed_stay_id"], how="left")
analytic_rad_w_text = analytic_rad_w_text.merge(f_xr_t,  on=["subject_id","hadm_id","ed_stay_id"], how="left")

for h in windows:
    hh = int(h)

    c_any = follow_count(follow[follow["poe_id"].notnull()], h, f"n_followup_rad_{hh}h")
    c_adv = follow_count(follow[is_adv_f], h, f"n_followup_advanced_{hh}h")
    c_xr  = follow_count(follow[is_xray_f], h, f"n_followup_xray_{hh}h")

    analytic_rad_w_text = analytic_rad_w_text.merge(c_any, on=["subject_id","hadm_id","ed_stay_id"], how="left")
    analytic_rad_w_text = analytic_rad_w_text.merge(c_adv, on=["subject_id","hadm_id","ed_stay_id"], how="left")
    analytic_rad_w_text = analytic_rad_w_text.merge(c_xr,  on=["subject_id","hadm_id","ed_stay_id"], how="left")

    analytic_rad_w_text[f"followup_any_within_{hh}h"] = (
        (analytic_rad_w_text["time_to_followup_any_hours"].notnull()) & (analytic_rad_w_text["time_to_followup_any_hours"] <= h)
    ).astype("int8")
    analytic_rad_w_text[f"followup_advanced_within_{hh}h"] = (
        (analytic_rad_w_text["time_to_followup_advanced_hours"].notnull()) & (analytic_rad_w_text["time_to_followup_advanced_hours"] <= h)
    ).astype("int8")
    analytic_rad_w_text[f"followup_xray_within_{hh}h"] = (
        (analytic_rad_w_text["time_to_followup_xray_hours"].notnull()) & (analytic_rad_w_text["time_to_followup_xray_hours"] <= h)
    ).astype("int8")

analytic_rad_w_text = analytic_rad_w_text.persist()

# ----------------------------
# Parquet dtype normalization (pyarrow)
# ----------------------------
analytic_rad_w_text["pain_min_pre_order"] = analytic_rad_w_text["pain_min_pre_order"].astype("float64")
analytic_rad_w_text["pain_max_pre_order"] = analytic_rad_w_text["pain_max_pre_order"].astype("float64")

last_cols = [c for c in analytic_rad_w_text.columns if c.endswith("_last")]
analytic_rad_w_text = analytic_rad_w_text.astype({c: "float64" for c in last_cols})

count_prefixes = ("n_rad_orders_", "n_advanced_orders_", "n_xray_orders_", "n_followup_")
count_cols = [c for c in analytic_rad_w_text.columns if c.startswith(count_prefixes) or c == "n_vitalsign_rows_pre_order"]
analytic_rad_w_text = analytic_rad_w_text.astype({c: "Int64" for c in count_cols})

lab_cols = [c for c in analytic_rad_w_text.columns if c.startswith("lab_")]
analytic_rad_w_text = analytic_rad_w_text.astype({c: "float64" for c in lab_cols})

analytic_rad_w_text = analytic_rad_w_text.persist()

# ----------------------------
# Save outputs
# ----------------------------
out_stay   = "rad_ed_stay.parquet"
out_order  = "rad_orders_ed_window.parquet"
out_note   = "rad_reports_ed_window.parquet"

analytic_rad_w_text.to_parquet(out_stay, write_index=False)
order_level_enriched.to_parquet(out_order, write_index=False)
note_level_out.to_parquet(out_note, write_index=False)

print("Saved:", out_stay, out_order, out_note)

# ----------------------------
# Sanity checks (hard asserts)
# ----------------------------
required_cols = [
    "index_order_subtype", "index_poe_id", "index_rad_ordertime",
    "exam_name", "exam_code", "cpt_code", "parent_note_id", "addendum_note_id",
    "rad_txt_first", "rad_txt_0_6h_all",
]
missing = [c for c in required_cols if c not in analytic_rad_w_text.columns]
assert len(missing) == 0, f"Missing required columns: {missing}"

print("has first rad report text:", analytic_rad_w_text["rad_txt_first"].notnull().mean().compute())
print("has any rad text:", analytic_rad_w_text["rad_txt_0_6h_all"].notnull().mean().compute())
print("has exam_name:", analytic_rad_w_text["exam_name"].notnull().mean().compute())
print("has index_order_subtype:", analytic_rad_w_text["index_order_subtype"].notnull().mean().compute())

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------------+------------+-------------+
| Merge columns        | left dtype | right dtype |
+----------------------+------------+-------------+
| ('itemid', 'itemid') | Int64      | int64       |
+----------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(


Saved: rad_ed_stay.parquet rad_orders_ed_window.parquet rad_reports_ed_window.parquet
has first rad report text: 0.5417701067896127
has any rad text: 0.5417701067896127
has exam_name: 0.5379625251211727
has index_order_subtype: 0.5789445166883398
